# Phase 5 — Legal Statute Identification with GQLA + a bidirectional head: the six experiments

**Idea.** The Phase-1 GQLA model is a *decoder* (causal): the state at token *t* has only seen tokens ≤ *t*. We keep it as a
pre-trained legal-text encoder and put **bidirectional** sequence models on top of its representations:

```
document → 512-token chunks (same window as pre-training)
        → GQLA hidden states (frozen, optionally top-k unfrozen)
        → [token-level BiGRU/BiLSTM]  → one vector per chunk
        → chunk-level BiGRU / BiLSTM / small BERT-style encoder
        → attention pooling → sigmoid per statute  (multi-label)
```

Two modes share everything after the GQLA:
* **`cached`** – GQLA is run once, each chunk stored as `[mean | max | last-token]` of its states; only the chunk-level bidirectional
  encoder is trained. Seconds per epoch → use this for the ablations.
* **`token`** – literal token-level BiRNN over GQLA states, then the chunk-level encoder. Slower, richer; `peft=dict(method=...)` adapts the GQLA itself (PEFT, partial or full fine-tuning); `unfreeze_top_k` still works.

**Macro-F1 is the primary metric**, so the notebook uses rare-statute-aware losses (`bce_pw`, `asl`), a force-at-least-one-statute rule, and
global / per-statute threshold tuning with an *honest* split-half check (tuning on dev and scoring on dev is optimistic).

**The six experiments of the Phase-5 brief (§11.2)** — all start from the Phase-1 GQLA checkpoint, no external model:

| # | experiment | technique | what is trained |
|---|---|---|---|
| E1 | `e1_lorma_plus` | **LoRMA+** (multiplicative low-rank, ACL Findings 2025) — *compulsory* | `A`,`B` of `(α/r·BA + I)·W0` on `W_UQ`, `W_UV` of the top 4 blocks |
| E2 | `e2_adalora` | **AdaLoRA** (SVD form, importance-based rank pruning) | `P`,`Λ`,`Q` on all 10 linears of the top 4 blocks, rank 12 → 6 |
| E3 | `e3_qlora` | **QLoRA** (frozen 4-bit NF4 GQLA + LoRA) | LoRA `A`,`B` on the top 4 blocks; all 10 blocks stored in 4 bit |
| E4 | `e4_adapter` | **Bottleneck adapters** (Houlsby: after attention and after the FFN) | adapters of the top 4 blocks |
| E5 | `e5_partial_ft` | **Partial fine-tuning** | the 3 top blocks + final norm |
| E6 | `e6_kd_student` | **Knowledge distillation** | student = first 5 GQLA blocks (frozen) + small BiGRU, teacher = ensemble of E1–E5 |

Optional extras (commented in RUN): additive LoRA, LoRMA-π, prompt tuning, full fine-tuning, the student without a teacher.
Cell *3b* proves that every PEFT method starts as the identity of the pre-trained GQLA. This notebook replaces the earlier patch cells
(fine-tuning wrapper, LoRA injection, checkpointing, distillation runner): the same ideas now live in one training loop with fp16 autocast,
gradient checkpointing and an out-of-memory fallback (`run_safe`).

> **All hyper-parameters in the RUN cell are placeholders** (4 epochs, ranks, learning rates, temperature …) chosen only so that the notebook runs end to end.
> Tuning them and interpreting the results is your job. Run **one experiment per cell first** and read the `s/batch -> ~N min per epoch` line before queueing the rest —
> token-mode experiments run all 10 GQLA blocks at every step. The brief also asks you to cover all four Phase-1 architectures and submit your best six overall.
> If you use an LLM for any of this, declare it as the course rules (§17.5) require. Put your Hugging Face token in a Colab *secret* named `HF_TOKEN`.

**Run order:** run all cells top to bottom (the new cell *3a* installs the fast GQLA attention and self-tests it before any features are extracted). Needs `GQLAModel`, `BPETokenizer`, `normalize_text` (cell 1 loads them from your Phase-1 notebook if they
are not already defined) and the Phase-1 checkpoint `best_model.pt`. Edit `GQLA_CFG` if you trained with different hyper-parameters.

In [1]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


In [2]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step


In [3]:
device = get_device()
print("Using device: ", device)

Using device:  cuda


In [4]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class GQLARotaryEmbedding(nn.Module):
    """
    Rotary embedding used only on the decoupled RoPE pathway.

    GQLA:
        q^R_t = RoPE_t(W_QR c^Q_t)
        k^R_t = RoPE_t(W_KR x_t)

    Input:
        x:
            [B, H, T, rope_dim]

        position_ids:
            [B, T]
    """

    def __init__(
        self,
        dim: int,
        base: float = 10000.0,
    ):
        super().__init__()

        if dim % 2 != 0:
            raise ValueError(
                f"RoPE dimension must be even, got {dim}"
            )

        self.dim = dim
        self.base = base

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    dim,
                    2,
                    dtype=torch.float32
                ) / dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

    @staticmethod
    def rotate_half(x):
        half = x.shape[-1] // 2

        x1 = x[..., :half]
        x2 = x[..., half:]

        return torch.cat(
            [-x2, x1],
            dim=-1
        )

    def forward(
        self,
        x,
        position_ids,
    ):
        """
        x:
            [B, H, T, dim]

        position_ids:
            [B, T]
        """

        # [B, T, dim/2]
        freqs = torch.einsum(
            "bt,d->btd",
            position_ids.float(),
            self.inv_freq
        )

        # [B, T, dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1
        )

        cos = emb.cos().unsqueeze(1)
        sin = emb.sin().unsqueeze(1)

        x_rope = x[..., :self.dim]

        x_rope = (
            x_rope * cos
            + self.rotate_half(x_rope) * sin
        )

        return x_rope



In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# GQLA
# ============================================================

class GroupQueryLatentAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
    ):
        super().__init__()

        if num_query_heads % num_kv_groups != 0:
            raise ValueError(
                "num_query_heads must be divisible by "
                "num_kv_groups"
            )

        if path not in {"gqa", "mqa_absorb"}:
            raise ValueError(
                "path must be either 'gqa' or 'mqa_absorb'"
            )

        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_groups = num_kv_groups
        self.head_dim = head_dim

        self.kv_latent_dim = kv_latent_dim
        self.rope_dim = rope_dim
        self.path = path # Add this line to store the path as an instance attribute

        self.heads_per_group = (
            num_query_heads // num_kv_groups
        )

        # ----------------------------------------------------
        # Q-side latent
        #
        # c_Q = W_DQ x
        # q_C = W_UQ c_Q
        # ----------------------------------------------------

        if q_latent_dim is None:
            # Paper explicitly notes that for models such as
            # LLaMA-3-8B and Qwen2.5-7B this factorization
            # collapses to W_Q directly.
            q_latent_dim = d_model

        self.q_latent_dim = q_latent_dim

        self.W_DQ = nn.Linear(
            d_model,
            q_latent_dim,
            bias=False
        )

        self.W_UQ = nn.Linear(
            q_latent_dim,
            num_query_heads * head_dim,
            bias=False
        )

        # ----------------------------------------------------
        # KV latent
        #
        # c_KV = W_DKV x
        # ----------------------------------------------------

        self.W_DKV = nn.Linear(
            d_model,
            kv_latent_dim,
            bias=False
        )

        # ----------------------------------------------------
        # GROUP indexed up-projections
        #
        # W_UK, W_UV:
        #
        # [g * d_h, r_KV]
        #
        # NOT:
        #
        # [h_q * d_h, r_KV]
        #
        # This group indexing is the key GQLA change.
        # ----------------------------------------------------

        self.W_UK = nn.Linear(
            kv_latent_dim,
            num_kv_groups * head_dim,
            bias=False
        )

        self.W_UV = nn.Linear(
            kv_latent_dim,
            num_kv_groups * head_dim,
            bias=False
        )

        # ----------------------------------------------------
        # Decoupled RoPE pathway
        #
        # q_R = RoPE(W_QR c_Q)
        # k_R = RoPE(W_KR x)
        # ----------------------------------------------------

        self.W_QR = nn.Linear(
            q_latent_dim,
            rope_dim,
            bias=False
        )

        self.W_KR = nn.Linear(
            d_model,
            rope_dim,
            bias=False
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.W_O = nn.Linear(
            num_query_heads * head_dim,
            d_model,
            bias=False
        )

        self.attn_dropout = nn.Dropout(dropout)

        self.rotary_emb = GQLARotaryEmbedding(
            dim=rope_dim,
            base=rope_base
        )

    # ========================================================
    # Helpers
    # ========================================================

    def _get_group_indices(self, device):
        """
        Maps each query head to its KV group.

        Example:

            hq = 8
            g  = 2

            query heads:

                0 1 2 3 | 4 5 6 7
                --------+--------
                  group 0 | group 1
        """

        return (
            torch.arange(
                self.num_query_heads,
                device=device
            )
            // self.heads_per_group
        )

    def _causal_mask(
        self,
        q_len,
        kv_len,
        past_len,
        device,
    ):
        """
        Correct causal mask when cached tokens exist.

        Example:

        past_len = 3
        q_len   = 2
        kv_len  = 5

        query position 3 can see:
            0 1 2 3

        query position 4 can see:
            0 1 2 3 4
        """

        return torch.triu(
            torch.ones(
                q_len,
                kv_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1 + past_len
        )

    # ========================================================
    # GQA decoding path
    # ========================================================

    def _forward_gqa(
        self,
        hidden_states,
        c_Q,
        c_KV,
        q_R,
        k_R_new,
        past_key_value,
        attention_mask,
        use_cache,
    ):
        """
        GQLA GQA path.

        Cache:

            K_C : [B, g, S, d_h]
            V_C : [B, g, S, d_h]
            K_R : [B, 1, S, d_R]

        This corresponds to the paper's per-group
        expanded GQA cache.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Content query
        # ----------------------------------------------------

        q_C = self.W_UQ(c_Q)

        q_C = q_C.view(
            B,
            T,
            self.num_query_heads,
            self.head_dim
        )

        q_C = q_C.transpose(1, 2)

        # ----------------------------------------------------
        # Group K/V
        # ----------------------------------------------------

        k_C = self.W_UK(c_KV)

        k_C = k_C.view(
            B,
            T,
            self.num_kv_groups,
            self.head_dim
        )

        k_C = k_C.transpose(1, 2)

        v_C = self.W_UV(c_KV)

        v_C = v_C.view(
            B,
            T,
            self.num_kv_groups,
            self.head_dim
        )

        v_C = v_C.transpose(1, 2)

        # ----------------------------------------------------
        # Cache
        #
        # IMPORTANT:
        # We cache GROUP K/V, not repeated query-head K/V.
        # ----------------------------------------------------

        if past_key_value is not None:

            past_k_C, past_v_C, past_k_R = (
                past_key_value
            )

            past_len = past_k_C.shape[2]

            k_C = torch.cat(
                [past_k_C, k_C],
                dim=2
            )

            v_C = torch.cat(
                [past_v_C, v_C],
                dim=2
            )

            k_R = torch.cat(
                [past_k_R, k_R_new],
                dim=2
            )

        else:

            past_len = 0

            k_R = k_R_new

        kv_len = k_C.shape[2]

        # ----------------------------------------------------
        # Position IDs
        # ----------------------------------------------------

        q_positions = torch.arange(
            past_len,
            past_len + T,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        k_positions = torch.arange(
            kv_len,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        # ----------------------------------------------------
        # RoPE
        # ----------------------------------------------------

        q_R = self.rotary_emb(
            q_R,
            q_positions
        )

        k_R = self.rotary_emb(
            k_R,
            k_positions
        )

        # ----------------------------------------------------
        # Repeat group K/V across query heads
        #
        # g = 2
        # hq = 8
        #
        # group 0 -> query heads 0,1,2,3
        # group 1 -> query heads 4,5,6,7
        # ----------------------------------------------------

        k_C_attn = k_C.repeat_interleave(
            self.heads_per_group,
            dim=1
        )

        v_C_attn = v_C.repeat_interleave(
            self.heads_per_group,
            dim=1
        )

        k_R_attn = k_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        q_R_attn = q_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        # ----------------------------------------------------
        # Construct complete Q/K
        #
        # q_i = [q_C_i ; q_R_i]
        #
        # k_i = [k_C_group(i) ; k_R]
        # ----------------------------------------------------

        q = torch.cat(
            [q_C, q_R_attn],
            dim=-1
        )

        k = torch.cat(
            [k_C_attn, k_R_attn],
            dim=-1
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        scale = math.sqrt(
            self.head_dim + self.rope_dim
        )

        attn_scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        ) / scale

        # ----------------------------------------------------
        # Causal mask
        # ----------------------------------------------------

        causal_mask = self._causal_mask(
            q_len=T,
            kv_len=kv_len,
            past_len=past_len,
            device=hidden_states.device
        )

        attn_scores = attn_scores.masked_fill(
            causal_mask.unsqueeze(0).unsqueeze(0),
            torch.finfo(attn_scores.dtype).min
        )

        # ----------------------------------------------------
        # External mask
        # ----------------------------------------------------

        if attention_mask is not None:

            attn_scores = (
                attn_scores + attention_mask
            )

        # ----------------------------------------------------
        # Softmax
        # ----------------------------------------------------

        attn_weights = F.softmax(
            attn_scores.float(),
            dim=-1
        ).to(attn_scores.dtype)

        attn_weights = self.attn_dropout(
            attn_weights
        )

        # ----------------------------------------------------
        # Values
        # ----------------------------------------------------

        attn_output = torch.matmul(
            attn_weights,
            v_C_attn
        )

        # [B,H,T,D] -> [B,T,H*D]

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                self.num_query_heads * self.head_dim
            )
        )

        output = self.W_O(
            attn_output
        )

        # ----------------------------------------------------
        # Cache only if explicitly requested
        #
        # DETACH is important.
        # ----------------------------------------------------

        if use_cache:

            new_cache = (
                k_C.detach(),
                v_C.detach(),
                k_R.detach()
            )

        else:

            new_cache = None

        return output, new_cache

    # ========================================================
    # MQA-absorb path
    # ========================================================

    def _forward_mqa_absorb(
        self,
        hidden_states,
        c_Q,
        c_KV,
        q_R,
        k_R_new,
        past_key_value,
        attention_mask,
        use_cache,
    ):
        """
        MLA-style MQA-absorb path.

        Cache:

            c_KV : [B, S, r_KV]
            k_R  : [B, S, d_R]

        This is the compact latent cache described in Eq. (3)
        of the paper.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Content query
        # ----------------------------------------------------

        q_C = self.W_UQ(c_Q)

        q_C = q_C.view(
            B,
            T,
            self.num_query_heads,
            self.head_dim
        )

        q_C = q_C.transpose(1, 2)

        # ----------------------------------------------------
        # Cache latent
        # ----------------------------------------------------

        if past_key_value is not None:

            past_c_KV, past_k_R = (
                past_key_value
            )

            past_len = past_c_KV.shape[1]

            c_KV = torch.cat(
                [past_c_KV, c_KV],
                dim=1
            )

            k_R = torch.cat(
                [
                    past_k_R,
                    k_R_new.squeeze(1)
                ],
                dim=1
            )

        else:

            past_len = 0

            k_R = k_R_new.squeeze(1)

        kv_len = c_KV.shape[1]

        # ----------------------------------------------------
        # Positions
        # ----------------------------------------------------

        q_positions = torch.arange(
            past_len,
            past_len + T,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        k_positions = torch.arange(
            kv_len,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        # ----------------------------------------------------
        # RoPE
        # ----------------------------------------------------

        q_R = self.rotary_emb(
            q_R,
            q_positions
        )

        k_R = self.rotary_emb(
            k_R.unsqueeze(1),
            k_positions
        )

        # ----------------------------------------------------
        # Absorb W_UK into Q
        #
        # q'_C = W_UK^T q_C
        # ----------------------------------------------------

        group_indices = self._get_group_indices(
            hidden_states.device
        )

        W_UK = self.W_UK.weight

        W_UK = W_UK.view(
            self.num_kv_groups,
            self.head_dim,
            self.kv_latent_dim
        )

        W_UK_heads = W_UK[
            group_indices
        ]

        q_latent = torch.einsum(
            "bhtd,hdr->bhtr",
            q_C,
            W_UK_heads
        )

        # ----------------------------------------------------
        # q = [W_UK^T q_C ; q_R]
        # k = [c_KV ; k_R]
        # ----------------------------------------------------

        q = torch.cat(
            [
                q_latent,
                q_R.expand(
                    -1,
                    self.num_query_heads,
                    -1,
                    -1
                )
            ],
            dim=-1
        )

        k_latent = c_KV.unsqueeze(1).expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        k_rope = k_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        k = torch.cat(
            [
                k_latent,
                k_rope
            ],
            dim=-1
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        scale = math.sqrt(
            self.kv_latent_dim + self.rope_dim
        )

        attn_scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        ) / scale

        causal_mask = self._causal_mask(
            q_len=T,
            kv_len=kv_len,
            past_len=past_len,
            device=hidden_states.device
        )

        attn_scores = attn_scores.masked_fill(
            causal_mask.unsqueeze(0).unsqueeze(0),
            torch.finfo(attn_scores.dtype).min
        )

        if attention_mask is not None:

            attn_scores = (
                attn_scores + attention_mask
            )

        attn_weights = F.softmax(
            attn_scores.float(),
            dim=-1
        ).to(attn_scores.dtype)

        attn_weights = self.attn_dropout(
            attn_weights
        )

        # ----------------------------------------------------
        # Value is directly c_KV
        #
        # v_hat = c_KV
        # ----------------------------------------------------

        latent_output = torch.matmul(
            attn_weights,
            c_KV.unsqueeze(1).expand(
                -1,
                self.num_query_heads,
                -1,
                -1
            )
        )

        # ----------------------------------------------------
        # Absorb W_UV into output
        #
        # o_i = W_UV_i o'_i
        # ----------------------------------------------------

        W_UV = self.W_UV.weight

        W_UV = W_UV.view(
            self.num_kv_groups,
            self.head_dim,
            self.kv_latent_dim
        )

        W_UV_heads = W_UV[
            group_indices
        ]

        attn_output = torch.einsum(
            "bhtr,hdr->bhtd",
            latent_output,
            W_UV_heads
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                self.num_query_heads * self.head_dim
            )
        )

        output = self.W_O(
            attn_output
        )

        # ----------------------------------------------------
        # Compact cache
        # ----------------------------------------------------

        if use_cache:

            new_cache = (
                c_KV.detach(),
                k_R.squeeze(1).detach()
            )

        else:

            new_cache = None

        return output, new_cache

    # ========================================================
    # Main forward
    # ========================================================

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        past_key_value=None,
        use_cache=False,
    ):
        """
        Parameters
        ----------
        hidden_states:
            [B, T, d_model]

        attention_mask:
            Optional additive attention mask.

        past_key_value:
            GQA path:
                (
                    k_C,
                    v_C,
                    k_R
                )

            MQA-absorb path:
                (
                    c_KV,
                    k_R
                )

            IMPORTANT:
                Set to None for every new independent sequence.

        use_cache:
            False during normal training.
            True during autoregressive generation.

        Returns
        -------
        output:
            [B, T, d_model]

        past_key_value:
            None when use_cache=False.
            Otherwise updated cache.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Q latent
        # ----------------------------------------------------

        c_Q = self.W_DQ(hidden_states)

        # ----------------------------------------------------
        # KV latent
        # ----------------------------------------------------

        c_KV = self.W_DKV(hidden_states)

        # ----------------------------------------------------
        # q_R
        #
        # [B,T,dR]
        # ->
        # [B,1,T,dR]
        # ----------------------------------------------------

        q_R = self.W_QR(
            c_Q
        ).view(B, T, 1,self.rope_dim).transpose(1, 2)

        # ----------------------------------------------------
        # k_R
        # ----------------------------------------------------

        k_R_new = self.W_KR(
            hidden_states
        ).view(B, T, 1, self.rope_dim).transpose(1, 2)

        # ----------------------------------------------------
        # Select decoding path
        # ----------------------------------------------------

        if self.path == "gqa":

            return self._forward_gqa(
                hidden_states=hidden_states,
                c_Q=c_Q,
                c_KV=c_KV,
                q_R=q_R,
                k_R_new=k_R_new,
                past_key_value=past_key_value,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )

        else:

            return self._forward_mqa_absorb(
                hidden_states=hidden_states,
                c_Q=c_Q,
                c_KV=c_KV,
                q_R=q_R,
                k_R_new=k_R_new,
                past_key_value=past_key_value,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )

In [6]:
class GQLABlock(nn.Module):

    def __init__(
        self,
        d_model: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
        expansion_factor: int = 4,
    ):
        super().__init__()

        self.attn = GroupQueryLatentAttention(
            d_model=d_model,
            num_query_heads=num_query_heads,
            num_kv_groups=num_kv_groups,
            head_dim=head_dim,
            kv_latent_dim=kv_latent_dim,
            rope_dim=rope_dim,
            q_latent_dim=q_latent_dim,
            dropout=dropout,
            rope_base=rope_base,
            path=path,
        )

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, expansion_factor * d_model),
            nn.GELU(),
            nn.Linear(expansion_factor * d_model, d_model),
            nn.Dropout(dropout),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        past_key_value=None,
        use_cache=False,
    ):

        # Self-attention
        attn_output, new_past_key_value = self.attn(
            hidden_states=self.norm1(hidden_states),
            attention_mask=attention_mask,
            past_key_value=past_key_value,
            use_cache=use_cache,
        )

        hidden_states = hidden_states + self.dropout1(attn_output)

        # Feed-forward
        ff_output = self.feed_forward(self.norm2(hidden_states))
        hidden_states = hidden_states + self.dropout2(ff_output)

        return hidden_states, new_past_key_value


class GQLAModel(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
        pad_token_id: int = 0,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_token_id,
        )

        self.layers = nn.ModuleList([
            GQLABlock(
                d_model=d_model,
                num_query_heads=num_query_heads,
                num_kv_groups=num_kv_groups,
                head_dim=head_dim,
                kv_latent_dim=kv_latent_dim,
                rope_dim=rope_dim,
                q_latent_dim=q_latent_dim,
                dropout=dropout,
                rope_base=rope_base,
                path=path,
            )
            for _ in range(n_layers)
        ])

        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        use_cache=False,
    ):

        B, T = input_ids.shape

        hidden_states = self.token_embedding(input_ids)

        new_past_key_values = []

        for i, layer in enumerate(self.layers):

            past_key_value = (
                past_key_values[i]
                if past_key_values
                else None
            )

            hidden_states, new_past_key_value = layer(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                past_key_value=past_key_value,
                use_cache=use_cache,
            )

            if use_cache:
                new_past_key_values.append(new_past_key_value)

        hidden_states = self.norm_final(hidden_states)
        logits = self.lm_head(hidden_states)

        if use_cache:
            return logits, new_past_key_values
        else:
            return logits

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = None,
        eos_token_id: int = None,
    ):
        self.eval()
        generated_ids = input_ids.clone()
        past_key_values = None

        for _ in range(max_new_tokens):
            current_input_ids = generated_ids[:, -1:].to(input_ids.device) if generated_ids.shape[1] > 0 else input_ids[:, :1].to(input_ids.device)

            logits, past_key_values = self.forward(
                current_input_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k_logits, top_k_indices = torch.topk(logits, top_k)
                probs = F.softmax(top_k_logits, dim=-1)
                sampled = torch.multinomial(probs, num_samples=1)
                next_token_id = top_k_indices.gather(-1, sampled)
            else:
                next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

            if eos_token_id is not None and (next_token_id == eos_token_id).all():
                break

        return generated_ids


In [7]:
# ============================================================
# 0. SETUP
# Run this notebook in the SAME Colab runtime as your Phase-1 notebook, or let
# load_definitions_from_notebook() pull the class definitions out of it.
# ============================================================
import os, json, math, time, random, copy, pickle
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

# ---- Hugging Face token: Colab secret or environment variable. Never hard-code it. ----
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token available:", bool(HF_TOKEN))

# ---- paths (edit if yours differ) ------------------------------------------
cache_dir  = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
ckpt_path  = os.path.join(cache_dir, "gqla_checkpoints", "gqla_baseline_large.pt")   # Phase-1 best model
tok_path   = os.path.join(cache_dir, "final_bpe_en.json")
phase1_nb  = "/content/drive/MyDrive/Colab Notebooks/CS781_Capstone_Project_decoding.ipynb"
p5_dir     = os.path.join(cache_dir, "phase5"); os.makedirs(p5_dir, exist_ok=True)

# ---- GQLA hyper-parameters: MUST equal what you trained with (your cell 44) ---
GQLA_CFG = dict(d_model=512, n_layers=10, num_query_heads=64, num_kv_groups=32,
                head_dim=64, kv_latent_dim=96, rope_dim=32, path="gqa")


def load_definitions_from_notebook(nb_path):
    """exec the *definition* cells (BPETokenizer, normalize_text, GQLA classes) of
    the Phase-1 notebook so this one does not have to duplicate ~900 lines."""
    nb = json.load(open(nb_path, encoding="utf-8"))
    wanted = ["class BPETokenizer", "def normalize_text", "class GQLARotaryEmbedding",
              "class GroupQueryLatentAttention", "class GQLAModel"]
    picked = []
    for m in wanted:
        for i, c in enumerate(nb["cells"]):
            src = "".join(c["source"])
            if c["cell_type"] == "code" and m in src and i not in picked:
                picked.append(i); break
        else:
            raise RuntimeError(f"could not find a cell containing {m!r} in {nb_path}")
    for i in sorted(picked):
        src = "".join(nb["cells"][i]["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        exec(compile(src, f"<phase1 cell {i}>", "exec"), globals())
    print("loaded definitions from cells", sorted(picked))


if not all(n in globals() for n in ("BPETokenizer", "normalize_text", "GQLAModel")):
    load_definitions_from_notebook(phase1_nb)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID = tokenizer.stoi["<pad>"]
print("tokenizer vocab:", len(tokenizer.itos))


def load_pretrained_gqla():
    m = GQLAModel(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **GQLA_CFG)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"])          # strict: fails loudly on a config mismatch
    return m.eval()


GQLA = load_pretrained_gqla()
print(f"GQLA loaded: {sum(p.numel() for p in GQLA.parameters())/1e6:.1f}M params")

Mounted at /content/drive
device: cuda | Tesla T4
HF token available: False
loaded definitions from cells [10, 22, 32, 33, 35]
tokenizer vocab: 32768
GQLA loaded: 103.9M params


In [8]:
# ============================================================
# 1. LOAD PHASE-5 (LSI) DATA
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


if "_REPO_FILES" not in globals():                       # one Hub API call per session (was one per split)
    _REPO_FILES = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-5 folder without hard-coding its exact name."""
    files = _REPO_FILES
    hits = [f for f in files if f.endswith("/" + basename) and ("lsi" in f.lower() or "phase_5" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 5, got {hits}. "
                           f"Set the path by hand. All files: {files[:40]}")
    return hits[0]


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                       # {id: {text, labels}}  (Phase-1 style)
        raw = [{**v, "id": k} for k, v in raw.items()]
    recs = []
    for r in raw:                                   # [{id, text, labels}]  (Phase-5 spec)
        recs.append(dict(id=r["id"], text=r["text"], labels=[int(l) for l in r.get("labels", [])]))
    return recs


# re-running this cell in the same session reuses what is already in memory (no re-download / re-parse)
if not all(k in globals() for k in ("train_recs", "dev_recs", "test_recs")):
    train_recs, dev_recs, test_recs = read_split("train.json"), read_split("dev.json"), read_split("test.json")
print(len(train_recs), len(dev_recs), len(test_recs), "train/dev/test documents")

# ---- label space: compact index <-> statute id --------------------------------
all_labels = sorted({l for r in train_recs + dev_recs for l in r["labels"]})
lab2idx = {l: i for i, l in enumerate(all_labels)}
idx2lab = {i: l for l, i in lab2idx.items()}
L = len(all_labels)


def multi_hot(recs):
    y = np.zeros((len(recs), L), dtype=np.float32)
    for i, r in enumerate(recs):
        for l in r["labels"]:
            y[i, lab2idx[l]] = 1.0
    return y


y_train, y_dev = multi_hot(train_recs), multi_hot(dev_recs)
freq = y_train.sum(0)
print(f"{L} statutes | labels/doc: train {y_train.sum(1).mean():.2f}")
print(f"train docs per statute: min {freq.min():.0f}  median {np.median(freq):.0f}  max {freq.max():.0f}")
print(f"statutes with <=5 train docs: {(freq <= 5).sum()}   in dev but never in train: "
      f"{((y_dev.sum(0) > 0) & (freq == 0)).sum()}")

# ---- tokenise once (full documents), cache to Drive -----------------------------
def encode_docs(recs, tag, bs=2000, workers=min(4, os.cpu_count() or 1)):
    path = os.path.join(p5_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(recs), bs):
        texts = [normalize_text(r["text"]) for r in recs[i:i + bs]]        # same normalisation as Phase 1
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(texts, add_special_tokens=True, num_workers=workers, chunk_size=32)]
    pickle.dump(out, open(path, "wb"))
    return out


if not all(k in globals() for k in ("enc_train", "enc_dev", "enc_test")):
    enc_train, enc_dev, enc_test = (encode_docs(train_recs, "train"), encode_docs(dev_recs, "dev"),
                                    encode_docs(test_recs, "test"))
lens = np.array([len(x) for x in enc_train])
print("train tokens/doc  median %d  p90 %d  p99 %d  max %d" % tuple(np.percentile(lens, [50, 90, 99, 100]).astype(int)))
print("-> 512-token chunks/doc  median %.1f  p90 %.1f" % tuple(np.percentile(np.ceil(lens / 512), [50, 90])))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

42750 10181 13019 train/dev/test documents
100 statutes | labels/doc: train 3.70
train docs per statute: min 272  median 679  max 11730
statutes with <=5 train docs: 0   in dev but never in train: 0
train tokens/doc  median 636  p90 4148  p99 12295  max 140121
-> 512-token chunks/doc  median 2.0  p90 9.0


In [9]:
# ============================================================
# Pure-numpy utilities: chunking, metrics, thresholds
# (no torch here on purpose, so they are easy to unit-test)
# ============================================================
import numpy as np


# ---------------- long-document chunking ----------------
def chunk_ids(ids, chunk_len=512, stride=512, min_tail=16):
    """Split one tokenised document into windows of <= chunk_len tokens.
    stride == chunk_len  -> the same non-overlapping windows used in Phase-1 pre-training.
    stride <  chunk_len  -> overlapping windows."""
    if len(ids) <= chunk_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + chunk_len])
        if s + chunk_len >= len(ids):
            break
        s += stride
    # a tiny trailing fragment carries almost no information
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="head_tail"):
    """Long-document representation strategy (one of the ablations)."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(f"unknown strategy {strategy!r}")


# ---------------- metrics ----------------
def _counts(y, pred):
    tp = (y * pred).sum(0)
    fp = ((1 - y) * pred).sum(0)
    fn = (y * (1 - pred)).sum(0)
    return tp, fp, fn


def f1_per_class(y, pred):
    tp, fp, fn = _counts(y, pred)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0), y.sum(0)


def compute_metrics(y, pred):
    """macro_f1      : mean F1 over statutes that occur in `y` (primary metric)
       macro_f1_all  : mean F1 over every column, absent statutes count as 0
       micro_f1, subset_acc : the two secondary metrics"""
    f1, sup = f1_per_class(y, pred)
    tp, fp, fn = _counts(y, pred)
    return dict(
        macro_f1=float(f1[sup > 0].mean()) if (sup > 0).any() else 0.0,
        macro_f1_all=float(f1.mean()),
        micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        subset_acc=float((y == pred).all(1).mean()),
    )


# ---------------- thresholding ----------------
THR_GRID = np.round(np.arange(0.05, 0.951, 0.025), 4)


def predict_labels(probs, thr, force_top1=True):
    """thr: scalar or per-class array [L]. Every document has >=1 statute, so if
    nothing clears the threshold we fall back to the arg-max statute."""
    pred = (probs >= thr).astype(np.float32)
    if force_top1:
        empty = np.where(pred.sum(1) == 0)[0]
        if len(empty):
            pred[empty, probs[empty].argmax(1)] = 1.0
    return pred


def tune_global_threshold(probs, y, grid=THR_GRID, force_top1=True):
    scores = [compute_metrics(y, predict_labels(probs, t, force_top1))["macro_f1"] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])


def tune_per_class_threshold(probs, y, global_thr, grid=THR_GRID, min_support=3, shrink=0.5):
    """Per-statute thresholds. Statutes with < min_support positives in the tuning
    set keep the global threshold (too little evidence to tune); the rest are
    shrunk towards it:  thr_c = shrink*best_c + (1-shrink)*global."""
    n, L = probs.shape
    thr = np.full(L, global_thr, dtype=np.float32)
    sup = y.sum(0)
    for c in range(L):
        if sup[c] < min_support:
            continue
        P = probs[:, c:c + 1] >= grid[None, :]
        yc = y[:, c:c + 1] > 0
        tp = (P & yc).sum(0)
        fp = (P & ~yc).sum(0)
        fn = (~P & yc).sum(0)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        best = grid[np.flatnonzero(f1 == f1.max())]           # ties -> closest to global
        best = best[np.argmin(np.abs(best - global_thr))]
        thr[c] = shrink * best + (1 - shrink) * global_thr
    return thr


def honest_threshold_eval(probs, y, min_support=2, shrink=0.5, seed=0):
    """Tuning thresholds on dev and scoring on the same dev is optimistic. This
    tunes on one random half of dev and scores on the other half (and swaps)."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(y))
    A, B = perm[: len(y) // 2], perm[len(y) // 2:]
    res = {"fixed_0.5": [], "global": [], "per_class": []}
    for tr, te in ((A, B), (B, A)):
        g, _ = tune_global_threshold(probs[tr], y[tr])
        pc = tune_per_class_threshold(probs[tr], y[tr], g, min_support=min_support, shrink=shrink)
        res["fixed_0.5"].append(compute_metrics(y[te], predict_labels(probs[te], 0.5))["macro_f1"])
        res["global"].append(compute_metrics(y[te], predict_labels(probs[te], g))["macro_f1"])
        res["per_class"].append(compute_metrics(y[te], predict_labels(probs[te], pc))["macro_f1"])
    return {k: float(np.mean(v)) for k, v in res.items()}


def rare_report(y_dev, pred, y_train, edges=((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9))):
    """Mean F1 on dev, bucketed by how many TRAIN documents carry the statute."""
    f1, sup = f1_per_class(y_dev, pred)
    tf = y_train.sum(0)
    rows = []
    for lo, hi in edges:
        m = (tf >= lo) & (tf <= hi) & (sup > 0)
        rows.append((f"{lo}-{hi if hi < 10**9 else 'inf'}", int(m.sum()),
                     float(f1[m].mean()) if m.any() else float("nan")))
    return rows

In [10]:
# ============================================================
# 2. MODEL:  GQLA (causal)  ->  BiGRU/BiLSTM over tokens  ->  BiGRU/BiLSTM/BERT-style encoder over chunks
#
# Why a bidirectional head?  GQLA is a decoder: the state at token t has only seen tokens <= t,
# so early tokens know nothing about the rest of the chunk. A bidirectional layer on top of the
# GQLA states lets every position see its left AND right context. Legal documents are far longer
# than the 512 tokens GQLA was trained on, so we do it hierarchically:
#
#   doc -> chunks of <=512 tokens -> [frozen GQLA states] -> token-level BiRNN -> chunk vector
#       -> chunk-level BiRNN (or small bidirectional Transformer) -> attention pooling -> sigmoid per statute
# ============================================================
from contextlib import nullcontext
from torch.utils.checkpoint import checkpoint as grad_checkpoint


def _pack_run(rnn, x, lengths):
    """Run an RNN over right-padded x with packing so the backward direction starts at the real end."""
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
    out, _ = rnn(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
    return out


def _make_rnn(kind, d_in, hid, layers, dropout):
    cls = {"gru": nn.GRU, "lstm": nn.LSTM}[kind]
    return cls(d_in, hid, layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)


class GQLABackbone(nn.Module):
    """Wraps the pre-trained GQLAModel and returns hidden states (no LM head).
       use_layer      : -1 = final layer (after norm_final); k>=0 = output of block k (parameter-free LayerNorm).
                        Intermediate layers are often better features than the last, next-token-specialised one.
                        k>=0 also gives a TRUNCATED GQLA of k+1 blocks (the distillation student).
       unfreeze_top_k : 0 = fully frozen; k>0 = fine-tune the top k USED blocks (+ final norm if use_layer=-1).
       PEFT methods (cell 'PEFT for GQLA') inject trainable modules into the used blocks and lower `first_grad_block`, the
       first block that must run with autograd. Blocks below it run under no_grad (no activation memory); blocks from it
       upwards are gradient-checkpointed.
       The GQLA stays in eval mode (no dropout) even when the wrapper is in train mode -> the fused attention path of cell 3a
       stays active. Causality makes right-padding safe: real tokens never see the pad positions."""

    def __init__(self, gqla, use_layer=-1, unfreeze_top_k=0, grad_ckpt=True):
        super().__init__()
        self.gqla, self.use_layer, self.grad_ckpt = gqla, use_layer, grad_ckpt
        self.d_model = gqla.token_embedding.embedding_dim
        n = len(gqla.layers)
        self.n_used = n if use_layer == -1 else use_layer + 1          # blocks that contribute to the output
        for p in gqla.parameters():
            p.requires_grad = False
        self.first_grad_block = self.n_used                             # = nothing trainable inside the GQLA
        self.soft_prompt, self.train_embed = None, False
        if unfreeze_top_k > 0:
            self.unfreeze_blocks(max(0, self.n_used - unfreeze_top_k))

    def device(self):
        return self.gqla.token_embedding.weight.device

    def unfreeze_blocks(self, first):
        for blk in self.gqla.layers[first:self.n_used]:
            for p in blk.parameters():
                p.requires_grad = True
        if self.use_layer == -1:
            for p in self.gqla.norm_final.parameters():
                p.requires_grad = True
        self.first_grad_block = min(self.first_grad_block, first)

    def train(self, mode=True):
        super().train(mode)
        self.gqla.eval()
        return self

    def forward(self, ids):                                   # ids [M,T] -> [M,T,d]
        g = self.gqla
        lo = min(self.first_grad_block, self.n_used)
        with torch.set_grad_enabled(self.train_embed and torch.is_grad_enabled()):
            h = g.token_embedding(ids)
        P = 0
        if self.soft_prompt is not None:                      # prompt tuning: learned vectors prepended to every chunk
            P = self.soft_prompt.size(0)
            h = torch.cat([self.soft_prompt.unsqueeze(0).expand(h.size(0), -1, -1).to(h.dtype), h], dim=1)
        with torch.no_grad() if lo > 0 else nullcontext():    # frozen lower blocks: no graph, no memory
            for blk in g.layers[:lo]:
                h, _ = blk(h)
        for blk in g.layers[lo:self.n_used]:                  # blocks that need gradients (empty when frozen)
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                h = grad_checkpoint(lambda x, b=blk: b(x)[0], h, use_reentrant=False)
            else:
                h, _ = blk(h)
        h = g.norm_final(h) if self.use_layer == -1 else F.layer_norm(h, (self.d_model,))
        return h[:, P:] if P else h


class TokenBiRNN(nn.Module):
    """Bidirectional RNN over the GQLA token states of ONE chunk -> a single chunk vector (mean+max pool)."""

    def __init__(self, d_in, hid=128, kind="gru", layers=1, dropout=0.1):
        super().__init__()
        self.ln, self.drop = nn.LayerNorm(d_in), nn.Dropout(dropout)
        self.rnn = _make_rnn(kind, d_in, hid, layers, dropout)
        self.out_dim = 4 * hid

    def forward(self, h, tok_mask):                           # [M,T,d], [M,T] bool
        lengths = tok_mask.sum(1)
        out = _pack_run(self.rnn, self.drop(self.ln(h)), lengths).float()     # pool in fp32 even under autocast
        m = tok_mask.unsqueeze(-1)
        mean = (out * m).sum(1) / lengths.unsqueeze(-1)
        mx = out.masked_fill(~m, -1e4).max(1).values
        return torch.cat([mean, mx], -1)


class ChunkSeqEncoder(nn.Module):
    """Contextualises the sequence of chunk vectors.
       kind: 'gru' | 'lstm'  -> bidirectional RNN
             'transformer'   -> small BERT-style bidirectional encoder (learned chunk-position embeddings)
             'none'          -> no cross-chunk modelling (ablation baseline)"""

    def __init__(self, d_in, hid=256, kind="gru", layers=1, dropout=0.3, max_chunks=64, heads=4):
        super().__init__()
        self.kind, D = kind, 2 * hid
        self.inp = nn.Sequential(nn.LayerNorm(d_in), nn.Dropout(dropout), nn.Linear(d_in, D), nn.GELU())
        if kind in ("gru", "lstm"):
            self.rnn = _make_rnn(kind, D, hid, layers, dropout)
        elif kind == "transformer":
            self.pos = nn.Embedding(max_chunks, D)
            layer = nn.TransformerEncoderLayer(D, heads, 4 * D, dropout, activation="gelu",
                                               batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        elif kind != "none":
            raise ValueError(kind)
        self.out_dim = D

    def forward(self, x, chunk_mask):                         # [B,N,d_in], [B,N] bool
        z = self.inp(x)
        if self.kind in ("gru", "lstm"):
            z = _pack_run(self.rnn, z, chunk_mask.sum(1))
        elif self.kind == "transformer":
            z = z + self.pos(torch.arange(z.size(1), device=z.device))
            z = self.enc(z, src_key_padding_mask=~chunk_mask)
        return z


class DocHead(nn.Module):
    """Chunk sequence -> one logit per statute.
       'attn'       : one attention over chunks, shared linear classifier (few parameters, safest for rare statutes)
       'label_attn' : every statute has its own attention query and weight vector, i.e. each statute picks the chunks
                      that matter for it (more capacity; self.last_attn [B,L,N] shows which chunk fired for which statute)
       'meanmax'    : masked mean+max pooling"""

    def __init__(self, dim, n_labels, pool="attn", dropout=0.3):
        super().__init__()
        self.pool = pool
        self.drop = nn.Dropout(dropout)
        self.last_attn = None
        if pool == "attn":
            self.att = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
            self.fc = nn.Linear(dim, n_labels)
        elif pool == "label_attn":
            self.Q = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.Q)
            self.W = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.W)
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif pool == "meanmax":
            self.fc = nn.Linear(2 * dim, n_labels)
        else:
            raise ValueError(pool)

    def forward(self, z, mask):                               # [B,N,D], [B,N]
        if self.pool == "attn":
            s = self.att(z).squeeze(-1).masked_fill(~mask, -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            return self.fc(self.drop((a.unsqueeze(-1) * z).sum(1)))
        if self.pool == "label_attn":
            s = torch.einsum("bnd,ld->bln", z, self.Q).masked_fill(~mask[:, None, :], -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            ctx = torch.einsum("bln,bnd->bld", a, z)
            return (self.drop(ctx) * self.W).sum(-1) + self.b
        m = mask.unsqueeze(-1)
        mean = (z * m).sum(1) / m.sum(1)
        mx = z.masked_fill(~m, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], -1)))


class GQLABiRNNClassifier(nn.Module):
    """MODE 'token': token-level BiRNN on GQLA states (literal 'BiRNN on top of GQLA representations').
       The GQLA forward is re-run every step, in micro-batches of chunks: frozen (no_grad), adapted by cfg.peft,
       or with its top blocks fine-tuned."""

    def __init__(self, gqla, n_labels, cfg):
        super().__init__()
        self.backbone = GQLABackbone(gqla, cfg.use_layer, cfg.unfreeze_top_k)
        # PEFT / fine-tuning method for this experiment (None = frozen backbone). See cell 'PEFT for GQLA'.
        self.peft = apply_peft(self.backbone, cfg.peft) if getattr(cfg, "peft", None) else None
        self.tok = TokenBiRNN(self.backbone.d_model, cfg.tok_hid, cfg.tok_rnn, 1, 0.1)
        self.chunk = ChunkSeqEncoder(self.tok.out_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers,
                                     cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)
        self.chunk_bs = cfg.gqla_chunk_bs

    def forward(self, ids, tok_mask, chunk_mask):             # [B,N,T], [B,N,T], [B,N]
        B, N, T = ids.shape
        flat_ids, flat_tm = ids[chunk_mask], tok_mask[chunk_mask]      # only real chunks: [M,T]
        vecs = [self.tok(self.backbone(flat_ids[i:i + self.chunk_bs]), flat_tm[i:i + self.chunk_bs])
                for i in range(0, flat_ids.size(0), self.chunk_bs)]
        vecs = torch.cat(vecs)
        x = vecs.new_zeros(B, N, vecs.size(-1))
        x[chunk_mask] = vecs
        return self.head(self.chunk(x, chunk_mask), chunk_mask)


class CachedFeatureClassifier(nn.Module):
    """MODE 'cached': GQLA frozen and run ONCE; every chunk is stored as [mean | max | last-token] of its GQLA states.
       Only the chunk-level bidirectional encoder + head are trained -> seconds per epoch, ideal for ablations.
       The 'last-token' vector is the one that has seen the whole chunk (causal), so it is a natural summary."""

    def __init__(self, in_dim, n_labels, cfg):
        super().__init__()
        self.chunk = ChunkSeqEncoder(in_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)

    def forward(self, feats, chunk_mask):
        return self.head(self.chunk(feats, chunk_mask), chunk_mask)

In [11]:
# ============================================================
# PEFT FOR GQLA  -  the adaptation techniques of the Phase-5 brief, applied to the Phase-1 GQLA decoder
#
#   method        technique (reference)                              what is trained
#   ------------  -------------------------------------------------  ------------------------------------------
#   lorma_plus    LoRMA+  (Bihany, Patel, Modi, ACL-Findings 2025)    B, A of  (a/r*BA + I) W0   [multiplicative]
#   lorma_pi      LoRMA-pi (same paper, permutation inflation)        B, A of  I_pi(BA) W0
#   adalora       AdaLoRA (Zhang et al., 2023)                        P, Lambda, Q (SVD form) + rank pruning
#   qlora         QLoRA   (Dettmers et al., 2023)                     LoRA on top of a frozen 4-bit NF4 GQLA
#   adapter       Bottleneck adapters (Houlsby et al., 2019)          down/up projections after attention AND after the FFN
#   lora          LoRA (Hu et al., 2022), additive                    B, A of  W0 + a/r*BA
#   prompt        Prompt tuning (Lester et al., 2021)                 P soft tokens prepended to every chunk
#   partial       Partial / layer-wise fine-tuning                    top-k GQLA blocks (+ final norm)
#   full          Full fine-tuning                                    every GQLA block incl. the embeddings
#
# A GQLA block owns eight attention projections  W_DQ, W_UQ, W_DKV, W_UK, W_UV, W_QR, W_KR, W_O  (no bias) and two feed-forward
# linears  ff1 = feed_forward[0] (512->2048), ff2 = feed_forward[2] (2048->512)  (with bias). LoRMA adapts the query and value
# up-projections W_UQ and W_UV, the GQLA counterparts of the paper's q and v; AdaLoRA, QLoRA and LoRA adapt all ten.
# By default the new modules live in the TOP `top_blocks` blocks: the blocks underneath run under no_grad (no activation
# memory), which is what makes token-mode training feasible on a T4. The LM head is never used or adapted.
#
# Use:  run_experiment(name, mode="token", peft=dict(method="lorma_plus", r=8), ...)   (see the RUN cell)
# ============================================================
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# NormalFloat-4 code book (Dettmers et al., 2023): quantiles of N(0,1) normalised to [-1, 1], with an exact zero.
NF4_TABLE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
             -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
             0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
             0.7229568362236023, 1.0]



ATTN_LINEARS = ("W_DQ", "W_UQ", "W_DKV", "W_UK", "W_UV", "W_QR", "W_KR", "W_O")
ALL_LINEARS = ATTN_LINEARS + ("ff1", "ff2")

PEFT_DEFAULTS = {
    "lorma_plus": dict(r=8, alpha=16, top_blocks=4, targets=("W_UQ", "W_UV")),
    # I_pi materialises a (d_out x d_out) matrix per module -> only the 512-wide outputs by default (W_UQ would need 4096 x 4096)
    "lorma_pi":   dict(r=8, top_blocks=4, targets=("W_DQ", "W_O")),
    "adalora":    dict(init_r=12, target_r=6, alpha=32, top_blocks=4, targets=ALL_LINEARS,
                       t_init=0.1, t_final=0.2, delta_t=10, beta=0.85, orth=0.1),
    "qlora":      dict(r=16, alpha=32, top_blocks=4, targets=ALL_LINEARS, quant_all_blocks=True, blocksize=64),
    "lora":       dict(r=8, alpha=16, top_blocks=4, targets=ALL_LINEARS),
    "adapter":    dict(bottleneck=64, top_blocks=4),
    "prompt":     dict(n_tokens=20),
    "partial":    dict(k=3),
    "full":       dict(),
}


def _owner(block, name):
    """(sub-module, attribute name) that owns the nn.Linear called `name` inside a GQLABlock"""
    if name in ATTN_LINEARS:
        return block.attn, name
    return block.feed_forward, {"ff1": "0", "ff2": "2"}[name]


class PEFTHandle:
    """Everything train_model needs to know about the chosen method."""

    def __init__(self, method, spec):
        self.method, self.spec = method, spec
        self.params = []              # NEW trainable parameters created by the method (they get cfg.lr_peft)
        self.mods = []                # injected modules
        self.allocator = None         # AdaLoRA rank allocator

    def param_ids(self):
        return {id(p) for p in self.params}

    def setup(self, total_steps):
        if self.allocator is not None:
            self.allocator.setup(total_steps)

    def extra_loss(self):
        return self.allocator.orth_loss() if self.allocator is not None else 0.0

    def after_backward(self, step):                 # gradients are un-scaled here, before clipping
        if self.allocator is not None:
            self.allocator.update_importance()

    def after_step(self, step):                     # optimizer step done
        if self.allocator is not None:
            self.allocator.after_step(step)

    def describe(self):
        n = sum(p.numel() for p in self.params)
        extra = ""
        if self.allocator is not None:
            r = self.allocator.effective_ranks()
            extra = f" | AdaLoRA ranks now: min {min(r)} / mean {sum(r)/len(r):.1f} / max {max(r)}"
        return f"PEFT '{self.method}': {n/1e6:.3f}M new parameters in {len(self.mods)} modules{extra}"


# ------------------------------------------------------------------ LoRMA
class LoRMALinear(nn.Module):
    """Low-Rank MULTIPLICATIVE Adaptation:  h = I(BA) W0 x     (W0: frozen [d x k],  B: [d x r],  A: [r x d])

    'plus' (LoRMA+): I+(BA) = alpha/r * BA + I_d.   B = 0, A random  ->  identity at init.
                     Evaluated in the cheap order  y + alpha/r * B (A y)  with y = W0 x  (no d x d matrix is built).
    'pi'   (LoRMA-pi): I_pi(BA) rotates row i of BA cyclically by i, which lifts a rank-r matrix to (almost) full rank.
                     Init B[:,0]=1, rest random; A[0,0]=1, rest 0  ->  I_pi(BA) = I_d.   (no alpha in this variant)"""

    def __init__(self, base, r=8, alpha=16, variant="plus"):
        super().__init__()
        d = base.out_features
        self.weight, self.bias = base.weight, base.bias             # frozen, share storage with the original layer (bias is NOT rotated)
        self.variant, self.r, self.scale = variant, r, alpha / r
        self.A, self.B = nn.Parameter(torch.empty(r, d)), nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            self.A.data[0, 0] = 1.0
            nn.init.normal_(self.B, std=0.02)
            self.B.data[:, 0] = 1.0
            idx = (torch.arange(d)[None, :] - torch.arange(d)[:, None]) % d     # M_rot[i, j] = M[i, (j - i) mod d]
            self.register_buffer("rot_idx", idx, persistent=False)

    def forward(self, x):
        y = F.linear(x, self.weight)                                # W0 x
        if self.variant == "plus":
            out = y + self.scale * F.linear(F.linear(y, self.A), self.B)
        else:
            M = torch.gather(self.B @ self.A, 1, self.rot_idx)      # I_pi(BA)  [d x d]
            out = F.linear(y, M.to(y.dtype))
        return out if self.bias is None else out + self.bias


# ------------------------------------------------------------------ LoRA (additive; the baseline the multiplicative LoRMA is compared with)
class LoRAAddLinear(nn.Module):
    """W0 x + b + alpha/r * B A x   (B = 0 at init -> identity)"""

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.weight, self.bias, self.scale = base.weight, base.bias, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        return F.linear(x, self.weight, self.bias) + self.scale * F.linear(F.linear(x, self.A), self.B)


# ------------------------------------------------------------------ AdaLoRA
class AdaLoRALinear(nn.Module):
    """W0 x + alpha/r_init * P diag(E) Q x   (SVD-style update; E = singular values, pruned by AdaLoRAAllocator)."""

    def __init__(self, base, r=12, alpha=32):
        super().__init__()
        d, k = base.out_features, base.in_features
        self.weight, self.bias, self.scale = base.weight, base.bias, alpha / r
        self.P = nn.Parameter(torch.randn(d, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, k) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))

    def forward(self, x):
        y = F.linear(x, self.weight, self.bias)
        return y + self.scale * F.linear(F.linear(x, self.Q) * self.E, self.P)


class AdaLoRAAllocator:
    """Importance-based rank allocation (Zhang et al., 2023, Sec. 3.2-3.3).
       sensitivity  I(w) = |w * dL/dw|, smoothed  I_bar <- b I_bar + (1-b) I,  U <- b U + (1-b)|I - I_bar|,  s(w) = I_bar * U
       triplet i    S_i = s(E_i) + mean_j s(P_ji) + mean_j s(Q_ij)
       every delta_t steps the globally lowest-scoring triplets are zeroed so that only b(t) remain, with the cubic
       schedule  b(t) = b_T + (b_0 - b_T) (1 - (t-t_i)/(T-t_i-t_f))^3.  In the last t_f steps the pattern is frozen.
       Orthogonality regulariser  orth * mean_modules( ||P^T P - I||_F + ||Q Q^T - I||_F )."""

    def __init__(self, mods, target_r, t_init, t_final, delta_t, beta, orth):
        self.mods, self.delta_t, self.beta, self.orth = mods, delta_t, beta, orth
        self.t_init_frac, self.t_final_frac = t_init, t_final
        self.b0 = sum(m.E.numel() for m in mods)
        self.bT = int(target_r * len(mods))
        self.ipt, self.unc, self.keep, self.total = {}, {}, None, None

    def setup(self, total_steps):
        self.total = total_steps
        self.ti, self.tf = int(self.t_init_frac * total_steps), int(self.t_final_frac * total_steps)
        self.keep = None

    def orth_loss(self):
        loss = 0.0
        for m in self.mods:
            I = torch.eye(m.P.size(1), device=m.P.device)
            loss = loss + torch.norm(m.P.t() @ m.P - I) + torch.norm(m.Q @ m.Q.t() - I)
        return self.orth * loss / len(self.mods)

    @torch.no_grad()
    def update_importance(self):
        b = self.beta
        for m in self.mods:
            for p in (m.P, m.E, m.Q):
                if p.grad is None:
                    continue
                s = torch.nan_to_num((p * p.grad).abs(), nan=0.0, posinf=0.0, neginf=0.0)
                k = id(p)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(p), torch.zeros_like(p)
                self.ipt[k].mul_(b).add_(s, alpha=1 - b)
                self.unc[k].mul_(b).add_((s - self.ipt[k]).abs(), alpha=1 - b)

    @torch.no_grad()
    def _scores(self):
        out = []
        for m in self.mods:
            sE = self.ipt.get(id(m.E), torch.zeros_like(m.E)) * self.unc.get(id(m.E), torch.zeros_like(m.E))
            sP = self.ipt.get(id(m.P), torch.zeros_like(m.P)) * self.unc.get(id(m.P), torch.zeros_like(m.P))
            sQ = self.ipt.get(id(m.Q), torch.zeros_like(m.Q)) * self.unc.get(id(m.Q), torch.zeros_like(m.Q))
            out.append(sE + sP.mean(0) + sQ.mean(1))                  # [r] per module
        return out

    @torch.no_grad()
    def _prune_to(self, budget):
        scores = self._scores()
        allv = torch.cat(scores)
        if budget >= allv.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        thr = torch.topk(allv, budget).values[-1]
        keep = [s >= thr for s in scores]
        for m, kp in zip(self.mods, keep):
            m.E.data.mul_(kp.to(m.E.dtype))                          # pruned singular values -> 0 (can be revived by gradient)
        return keep

    def budget_at(self, step):
        mul = 1 - (step - self.ti) / max(1, self.total - self.tf - self.ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    @torch.no_grad()
    def after_step(self, step):
        if self.total is None or step <= self.ti:
            return                                                    # warm-up at full rank
        if step > self.total - self.tf:                              # final phase: fixed rank pattern
            if self.keep is None:
                self.keep = self._prune_to(self.bT)
            for m, kp in zip(self.mods, self.keep):
                m.E.data.mul_(kp.to(m.E.dtype))
        elif step % self.delta_t == 0:
            self._prune_to(self.budget_at(step))

    def effective_ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]


# ------------------------------------------------------------------ QLoRA (NF4 base + LoRA)
class NF4Linear(nn.Module):
    """Frozen weight stored as 4-bit NormalFloat codes (two per byte) + one fp16 absmax per `blocksize` weights,
       de-quantised on the fly in the forward pass; optional LoRA update  W x + alpha/r * B A x  (A, B in 16/32 bit).
       Simplifications w.r.t. the QLoRA paper: no double quantisation of the absmax constants, no paged optimiser."""

    def __init__(self, base, r=0, alpha=16, blocksize=64):
        super().__init__()
        self.bias = base.bias                                         # frozen, kept in its original precision
        w = base.weight.data.detach().float()
        self.out_features, self.in_features, self.blocksize, self.r = base.out_features, base.in_features, blocksize, r
        flat = w.reshape(-1)
        n = flat.numel()
        assert n % blocksize == 0 and (n // 2) * 2 == n
        blk = flat.view(-1, blocksize)
        absmax = blk.abs().amax(1).clamp_min(1e-8)
        normed = (blk / absmax[:, None]).reshape(-1)
        table = torch.tensor(NF4_TABLE, device=w.device)
        codes = torch.empty(n, dtype=torch.uint8, device=w.device)
        for s in range(0, n, 1 << 20):                                # nearest code-book entry, in slices
            codes[s:s + (1 << 20)] = (normed[s:s + (1 << 20), None] - table[None, :]).abs().argmin(1).to(torch.uint8)
        self.register_buffer("qweight", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("table", table, persistent=False)
        self.n = n
        if r > 0:
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, self.in_features))
            self.B = nn.Parameter(torch.zeros(self.out_features, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def dequantize(self, dtype):
        codes = torch.stack([(self.qweight >> 4).long(), (self.qweight & 15).long()], 1).reshape(-1)
        w = self.table[codes].view(-1, self.blocksize) * self.absmax.float()[:, None]
        return w.reshape(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        y = F.linear(x, self.dequantize(x.dtype), self.bias)
        if self.r > 0:
            y = y + self.scale * F.linear(F.linear(x, self.A), self.B)
        return y


# ------------------------------------------------------------------ bottleneck adapter
class BottleneckAdapter(nn.Module):
    """y + W_up gelu(W_down y), W_up = 0 at init (identity). Houlsby placement: one adapter after the attention sub-layer and
       one after the FFN sub-layer of every adapted block (both applied before the residual connection)."""

    def __init__(self, d, bottleneck):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


def _block_forward(self, hidden_states, attention_mask=None, past_key_value=None, use_cache=False):
    """GQLABlock.forward with optional Houlsby adapters. Without adapters it is exactly the original forward."""
    a, new_past = self.attn(hidden_states=self.norm1(hidden_states), attention_mask=attention_mask,
                            past_key_value=past_key_value, use_cache=use_cache)
    ad = getattr(self, "adapter_attn", None)
    if ad is not None:
        a = ad(a)
    hidden_states = hidden_states + self.dropout1(a)
    f = self.feed_forward(self.norm2(hidden_states))
    ad = getattr(self, "adapter_ff", None)
    if ad is not None:
        f = ad(f)
    hidden_states = hidden_states + self.dropout2(f)
    return hidden_states, new_past


if not hasattr(GQLABlock, "_orig_forward"):          # patch once, so re-running the cell never patches a patch
    GQLABlock._orig_forward = GQLABlock.forward
    GQLABlock.forward = _block_forward


# ------------------------------------------------------------------ injection
def apply_peft(bb, spec):
    """Inject the method described by spec = dict(method=..., **overrides) into backbone `bb` (a GQLABackbone that already
       froze all pre-trained weights). Returns a PEFTHandle. Call on a deep copy of the GQLA."""
    spec = dict(spec)
    method = spec.pop("method")
    c = {**PEFT_DEFAULTS[method], **spec}
    h = PEFTHandle(method, c)
    layers = bb.gqla.layers
    n = bb.n_used
    first = 0 if c.get("top_blocks") is None else max(0, n - c["top_blocks"])

    if method == "partial":
        bb.unfreeze_blocks(max(0, n - c["k"]))
    elif method == "full":
        bb.unfreeze_blocks(0)
        bb.train_embed = True
        bb.gqla.token_embedding.weight.requires_grad = True
    elif method == "prompt":
        emb = bb.gqla.token_embedding.weight
        ids = torch.randint(5, emb.size(0), (c["n_tokens"],))        # init from random vocabulary rows (Lester et al.)
        bb.soft_prompt = nn.Parameter(emb.data[ids].clone().float())
        bb.first_grad_block = 0                                      # the prompt sits below every block
        h.params.append(bb.soft_prompt)
    elif method == "adapter":
        for i in range(first, n):
            blk = layers[i]
            blk.adapter_attn = BottleneckAdapter(bb.d_model, c["bottleneck"]).to(bb.device())
            blk.adapter_ff = BottleneckAdapter(bb.d_model, c["bottleneck"]).to(bb.device())
            for ad in (blk.adapter_attn, blk.adapter_ff):
                h.mods.append(ad)
                h.params += list(ad.parameters())
        bb.first_grad_block = min(bb.first_grad_block, first)
    else:                                                            # methods that replace attention / FFN linears
        quant_from = 0 if (method == "qlora" and c["quant_all_blocks"]) else first
        for i in range(quant_from, n):
            for name in c["targets"]:
                owner, attr = _owner(layers[i], name)
                base = getattr(owner, attr)
                adapted = i >= first
                if method in ("lorma_plus", "lorma_pi"):
                    new = LoRMALinear(base, c["r"], c.get("alpha", 16), "plus" if method == "lorma_plus" else "pi")
                    h.params += [new.A, new.B]
                elif method == "lora":
                    new = LoRAAddLinear(base, c["r"], c["alpha"])
                    h.params += [new.A, new.B]
                elif method == "adalora":
                    new = AdaLoRALinear(base, c["init_r"], c["alpha"])
                    h.params += [new.P, new.Q, new.E]
                elif method == "qlora":
                    new = NF4Linear(base, c["r"] if adapted else 0, c["alpha"], c["blocksize"])
                    if adapted:
                        h.params += [new.A, new.B]
                else:
                    raise ValueError(method)
                new = new.to(base.weight.device)
                setattr(owner, attr, new)
                if method != "qlora" or adapted:
                    h.mods.append(new)
        bb.first_grad_block = min(bb.first_grad_block, first)
        if method == "adalora":
            h.allocator = AdaLoRAAllocator(h.mods, c["target_r"], c["t_init"], c["t_final"],
                                           c["delta_t"], c["beta"], c["orth"])
    return h


# ------------------------------------------------------------------ size / cost accounting
def gqla_prefix_footprint(gqla, n_used):
    """(#params, bytes) of the part of the GQLA that a deployed pipeline needs: embedding + the first n_used blocks (+ final norm
       if all are used) + their PEFT modules, 4-bit buffers included. The LM head is never needed."""
    n_total = len(gqla.layers)
    params = nbytes = 0
    for name, p in gqla.named_parameters():
        mt = re.match(r"layers\.(\d+)\.", name)
        if mt and int(mt.group(1)) >= n_used:
            continue
        if name.startswith("lm_head"):
            continue
        if name.startswith("norm_final") and n_used < n_total:
            continue
        params += p.numel(); nbytes += p.numel() * p.element_size()
    for name, m in gqla.named_modules():
        mt = re.match(r"layers\.(\d+)\.", name)
        if isinstance(m, NF4Linear) and not (mt and int(mt.group(1)) >= n_used):
            params += m.n; nbytes += m.qweight.numel() + m.absmax.numel() * 2
    return params, nbytes


def model_footprint(model, mode, n_used_cached=None):
    """Deployable pipeline = GQLA prefix that is used + everything trained on top of it."""
    if mode == "token":
        bb = model.backbone
        p0, b0 = gqla_prefix_footprint(bb.gqla, bb.n_used)
        head = [(n, p) for n, p in model.named_parameters() if not n.startswith("backbone.")]
        extra = [bb.soft_prompt] if bb.soft_prompt is not None else []
    else:
        p0, b0 = gqla_prefix_footprint(GQLA, n_used_cached)
        head, extra = list(model.named_parameters()), []
    pn = sum(p.numel() for _, p in head) + sum(p.numel() for p in extra)
    bn = sum(p.numel() * p.element_size() for _, p in head) + sum(p.numel() * p.element_size() for p in extra)
    return (p0 + pn) / 1e6, (b0 + bn) / 1e6


In [12]:
# ============================================================
# 3a. SPEED PATCH for GQLA feature extraction (run once, after cells 4-6 and 10)
#
# Why extraction was slow (all in the frozen GQLA forward):
#   1. Attention built the full [B, 64 heads, 512, 512] fp32 score matrix in EVERY
#      layer (~1 GB at B=16), then masked_fill / softmax-float / matmul each made
#      another full copy -> memory-bandwidth bound.
#   2. Everything ran in fp32 (no tensor-core use).
#   3. Fixed batch of 16 chunks, per-chunk python padding, a GPU->CPU sync per batch.
# Fixes: fused SDPA attention (causal, no TxT matrix), fp16/bf16 autocast with RoPE
# kept in fp32, token-budget batching, one preallocated output tensor, local-disk
# cache in front of Drive, throughput/ETA logging.
# ============================================================
import time, shutil
import torch, torch.nn.functional as F

FAST = {"on": True}          # set FAST["on"] = False to fall back to the original attention

# keep the originals (only once, so re-running this cell never patches a patch)
if not hasattr(GroupQueryLatentAttention, "_orig_forward_gqa"):
    GroupQueryLatentAttention._orig_forward_gqa = GroupQueryLatentAttention._forward_gqa


# ---- RoPE always in fp32 ------------------------------------------------------
# Under autocast the original einsum("bt,d->btd") runs in fp16: position 500 * freq
# is rounded to a 0.25 grid -> corrupted rotary angles. Compute angles in fp32.
def _rope_fp32(self, x, position_ids):
    with torch.autocast(device_type=x.device.type, enabled=False):
        freqs = position_ids.float().unsqueeze(-1) * self.inv_freq.float()   # [B,T,d/2]
        emb = torch.cat([freqs, freqs], dim=-1)
        cos, sin = emb.cos().unsqueeze(1), emb.sin().unsqueeze(1)
        xr = x[..., :self.dim].float()
        out = xr * cos + self.rotate_half(xr) * sin
    return out.to(x.dtype)

GQLARotaryEmbedding.forward = _rope_fp32


# ---- fused causal attention (same maths as _forward_gqa without cache/mask) -------
def _fast_forward_gqa(self, hidden_states, c_Q, c_KV, q_R, k_R_new,
                      past_key_value, attention_mask, use_cache):
    if (not FAST["on"] or past_key_value is not None or attention_mask is not None
            or use_cache or (self.training and self.attn_dropout.p > 0)):
        return self._orig_forward_gqa(hidden_states, c_Q, c_KV, q_R, k_R_new,
                                      past_key_value, attention_mask, use_cache)
    B, T, _ = hidden_states.shape
    H, G, D = self.num_query_heads, self.num_kv_groups, self.head_dim
    q_C = self.W_UQ(c_Q).view(B, T, H, D).transpose(1, 2)
    k_C = self.W_UK(c_KV).view(B, T, G, D).transpose(1, 2)
    v_C = self.W_UV(c_KV).view(B, T, G, D).transpose(1, 2)
    pos = torch.arange(T, device=hidden_states.device).unsqueeze(0).expand(B, -1)
    q_R = self.rotary_emb(q_R, pos)
    k_R = self.rotary_emb(k_R_new, pos)
    q = torch.cat([q_C, q_R.expand(-1, H, -1, -1)], dim=-1)
    k = torch.cat([k_C.repeat_interleave(self.heads_per_group, 1),
                   k_R.expand(-1, H, -1, -1)], dim=-1)
    v = v_C.repeat_interleave(self.heads_per_group, 1)
    # default scale = 1/sqrt(head_dim + rope_dim) == the original `scale`
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return self.W_O(o.transpose(1, 2).reshape(B, T, H * D)), None

GroupQueryLatentAttention._forward_gqa = _fast_forward_gqa


# ---- mixed precision choice ----------------------------------------------------
def pick_amp_dtype():
    if device.type != "cuda":
        return None
    # bf16 only on Ampere+ (a T4 would emulate it slowly) -> fp16 on T4/V100
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

AMP_DTYPE = pick_amp_dtype()
print("fast attention patch installed | AMP dtype:", AMP_DTYPE)


# ---- self-test: run this once. It proves the patch did not change the features ----
@torch.no_grad()
def check_fast_path(n=8, T=512):
    docs = [d for d in enc_train if len(d) >= T][:n]
    assert docs, "no training document has >= 512 tokens"
    ids = torch.as_tensor(np.stack([d[:T] for d in docs]), dtype=torch.long, device=device)
    bb = GQLABackbone(GQLA.to(device), -1, 0).to(device).eval()

    def timed(fn):
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time(); out = fn().float()
        if device.type == "cuda": torch.cuda.synchronize()
        return out, time.time() - t

    FAST["on"] = False; ref, t_ref = timed(lambda: bb(ids))
    FAST["on"] = True;  f32, t_f32 = timed(lambda: bb(ids))

    def amp_run():
        with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=AMP_DTYPE is not None):
            return bb(ids)
    f16, t_f16 = timed(amp_run)

    cos = lambda a, b: F.cosine_similarity(a.reshape(-1, a.size(-1)), b.reshape(-1, b.size(-1)), dim=-1).min().item()
    print(f"original fp32      : {t_ref*1000:7.0f} ms")
    print(f"SDPA fp32          : {t_f32*1000:7.0f} ms | max|diff| {(ref-f32).abs().max():.2e} | min cos {cos(ref, f32):.6f}")
    print(f"SDPA + autocast    : {t_f16*1000:7.0f} ms | max|diff| {(ref-f16).abs().max():.2e} | min cos {cos(ref, f16):.6f}")
    print("NaN/Inf in AMP output:", bool((~torch.isfinite(f16)).any()))
    assert (ref - f32).abs().max() < 1e-3, "SDPA path does not match the original attention!"
    if cos(ref, f16) < 0.99 or not torch.isfinite(f16).all():
        print("WARNING: mixed precision is hurting the features -> use amp=False in extract_chunk_features")

check_fast_path()


fast attention patch installed | AMP dtype: torch.float16
original fp32      :    1103 ms
SDPA fp32          :    1174 ms | max|diff| 8.34e-06 | min cos 1.000000
SDPA + autocast    :     197 ms | max|diff| 7.66e-03 | min cos 0.999999
NaN/Inf in AMP output: False


In [13]:
# ============================================================
# 3b. SELF-TEST of the PEFT modules -- run once, takes seconds. Every method must start as the IDENTITY on the pre-trained
#     GQLA (so training begins from the Phase-1 model): features with the method injected == features without it.
#     Exceptions by design: QLoRA (4-bit rounding changes the weights slightly) and prompt tuning (adds tokens) -- they are
#     printed, not asserted.
# ============================================================
@torch.no_grad()
def check_peft_identity():
    docs = [d for d in enc_train if len(d) >= 200][:4]
    ids = torch.as_tensor(np.stack([d[:200] for d in docs]), dtype=torch.long, device=device)

    ref = GQLABackbone(GQLA.to(device), -1, 0).eval()(ids).float()
    print(f"{'method':12s} {'new params':>11s} {'max|diff| vs pre-trained':>26s}")
    for spec in (dict(method="lorma_plus"), dict(method="lorma_pi"), dict(method="lora"), dict(method="adalora"),
                 dict(method="adapter"), dict(method="qlora"), dict(method="prompt")):
        bb = GQLABackbone(copy.deepcopy(GQLA), -1, 0).to(device).eval()
        h = apply_peft(bb, spec)
        d = float((bb(ids).float() - ref).abs().max())
        n = sum(p.numel() for p in h.params)
        approx = spec["method"] in ("qlora", "prompt")
        print(f"{spec['method']:12s} {n:11,d} {d:26.2e}{'' if approx else ('   OK' if d < 1e-4 else '   <-- NOT identity!')}")
        if not approx:
            assert d < 1e-4, f"{spec['method']} is not the identity at initialisation"
        del bb
    if device.type == "cuda":
        torch.cuda.empty_cache()


check_peft_identity()


method        new params   max|diff| vs pre-trained
lorma_plus       393,216                   0.00e+00   OK
lorma_pi          65,536                   0.00e+00   OK
lora             683,008                   0.00e+00   OK
adalora        1,024,992                   0.00e+00   OK
adapter          528,896                   0.00e+00   OK
qlora          1,366,016                   9.81e-01
prompt            10,240                   4.07e+00


In [14]:
# ============================================================
# 3. DATASETS, COLLATE, GQLA FEATURE EXTRACTION
# ============================================================
# ---- session-level caches: every split is chunked / packed / read from disk ONCE, then reused by all experiments ----
_ITEMS_MEM = {}     # (split, chunk_len, stride, max_chunks, strategy) -> per-doc chunk lists
_PACK_MEM = {}      # same key -> per-doc padded int32 id tensors (built lazily, only for mode='token')
_FEAT_MEM = {}      # feature file name -> list of per-doc fp16 feature tensors


def _pack_doc(chunks):
    """One document's chunks -> right-padded int32 tensor [N, T] + lengths [N] (built once, reused every epoch)."""
    lens = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lens.max())), PAD_ID, dtype=torch.int32)
    for n, c in enumerate(chunks):
        ids[n, :len(c)] = torch.from_numpy(np.asarray(c, dtype=np.int32))
    return ids, lens


def _chunk_key(split, cfg):
    return (split, cfg.chunk_len, cfg.chunk_stride, cfg.max_chunks, cfg.chunk_strategy)


def chunked_items(split, enc_docs, cfg):
    key = _chunk_key(split, cfg)
    if key not in _ITEMS_MEM:
        t0 = time.time()
        _ITEMS_MEM[key] = [select_chunks(chunk_ids(d, cfg.chunk_len, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy)
                           for d in enc_docs]
        print(f"  chunked '{split}' once ({len(enc_docs):,} docs, {time.time()-t0:.1f}s) -> reused by later experiments")
    return _ITEMS_MEM[key]


def packed_items(split, enc_docs, cfg):
    key = _chunk_key(split, cfg)
    if key not in _PACK_MEM:
        _PACK_MEM[key] = [_pack_doc(c) for c in chunked_items(split, enc_docs, cfg)]
    return _PACK_MEM[key]


class ChunkedDocs(Dataset):
    """Each document -> list of <=512-token chunks (after the chosen long-document strategy).
       Pass `split` to reuse the session cache instead of re-chunking every experiment."""

    def __init__(self, enc_docs, y, cfg, split=None, pack=True):
        if split is not None:
            self.items = chunked_items(split, enc_docs, cfg)
            self.packed = packed_items(split, enc_docs, cfg) if pack else None
        else:
            self.items = [select_chunks(chunk_ids(d, cfg.chunk_len, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy)
                          for d in enc_docs]
            self.packed = None
        self.y = y

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        doc = self.packed[i] if self.packed is not None else _pack_doc(self.items[i])
        return i, doc, (None if self.y is None else self.y[i])


class FeatDocs(Dataset):
    def __init__(self, feats, y):
        self.feats, self.y = feats, y

    def __len__(self): return len(self.feats)

    def __getitem__(self, i): return i, self.feats[i], (None if self.y is None else self.y[i])


def _stack_y(batch):
    return None if batch[0][2] is None else torch.from_numpy(np.stack([b[2] for b in batch]))


def collate_tokens(batch):                                   # items are pre-packed (ids [N,T], lens [N]) per document
    B = len(batch)
    N = max(d[0].size(0) for _, d, _ in batch)
    T = max(d[0].size(1) for _, d, _ in batch)
    ids = torch.full((B, N, T), PAD_ID, dtype=torch.long)
    cm = torch.zeros(B, N, dtype=torch.bool)
    lens = torch.zeros(B, N, dtype=torch.long)
    for b, (_, (d_ids, d_lens), _) in enumerate(batch):
        n, t = d_ids.shape
        ids[b, :n, :t] = d_ids                                   # one copy per document instead of one per chunk
        lens[b, :n] = d_lens
        cm[b, :n] = True
    tm = torch.arange(T)[None, None, :] < lens[:, :, None]
    return dict(ids=ids, tok_mask=tm, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))   # idx: doc index (KD teacher logits)


def collate_feats(batch):
    B, N, F_ = len(batch), max(f.size(0) for _, f, _ in batch), batch[0][1].size(1)
    x = torch.zeros(B, N, F_)
    cm = torch.zeros(B, N, dtype=torch.bool)
    for b, (_, f, _) in enumerate(batch):
        x[b, :f.size(0)] = f.float()
        cm[b, :f.size(0)] = True
    return dict(feats=x, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))


@torch.no_grad()
def extract_chunk_features(gqla, ds, tok_budget=32768, max_bs=256, use_layer=-1, amp=True, log_every=25):
    """Run the frozen GQLA once over every chunk -> fp16 [N_i, 3*d] per document.
    Batches are built from a TOKEN budget over length-sorted chunks (short chunks -> bigger
    batches), under fp16/bf16 autocast, with fused causal attention (see cell 3a).
    Right-padding is safe: attention is causal, so real tokens never see the pad positions."""
    bb = GQLABackbone(gqla, use_layer, 0).to(device).eval()
    flat = [c for chunks in ds.items for c in chunks]                 # doc-major order
    doc_len = [len(chunks) for chunks in ds.items]
    n, d = len(flat), bb.d_model
    lens = np.fromiter((len(c) for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")                           # similar lengths -> little padding
    out = torch.empty(n, 3 * d, dtype=torch.float16)                  # one preallocated buffer
    use_amp = amp and AMP_DTYPE is not None
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens | amp={AMP_DTYPE if use_amp else 'off'} | budget {tok_budget} tok/batch")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            arr[j, :lens[i]] = flat[i]
        ids = torch.from_numpy(arr).to(device, non_blocking=True)
        ln = torch.from_numpy(lens[sel]).to(device)
        with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=use_amp):
            h = bb(ids)
        h = h.float()
        m = (torch.arange(T, device=device)[None, :] < ln[:, None]).unsqueeze(-1)
        mean = (h * m).sum(1) / ln[:, None]
        mx = h.masked_fill(~m, -1e4).max(1).values
        last = h[torch.arange(len(sel), device=device), ln - 1]
        out[torch.from_numpy(sel)] = torch.cat([mean, mx, last], -1).half().cpu()

    def safe_run(sel):                                                # halve the batch on CUDA OOM
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache()
            mid = len(sel) // 2
            safe_run(sel[:mid]); safe_run(sel[mid:])

    t0, done_tok, nb, s = time.time(), 0, 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        sel = order[s:e]
        safe_run(sel)
        done_tok += int(lens[sel].sum()); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            dt = time.time() - t0
            eta = dt * (int(lens.sum()) - done_tok) / max(done_tok, 1)
            print(f"  features {s:,}/{n:,} chunks | {done_tok/dt:,.0f} tok/s | elapsed {dt/60:.1f} min | ETA {eta/60:.1f} min")

    feats, p = [], 0
    for k in doc_len:                                                 # per-doc views of the one buffer
        feats.append(out[p:p + k]); p += k
    return feats


def get_features(split, enc_docs, cfg):
    sig = f"{split}_L{cfg.chunk_len}_S{cfg.chunk_stride}_K{cfg.max_chunks}_{cfg.chunk_strategy}_layer{cfg.use_layer}"
    name = f"feat_{sig}.pt"
    drive_path = os.path.join(p5_dir, name)
    local_dir = "/content/feat_cache"; os.makedirs(local_dir, exist_ok=True)   # local disk is much faster than Drive
    local_path = os.path.join(local_dir, name)
    if name in _FEAT_MEM:                                                      # already in RAM this session
        return _FEAT_MEM[name]
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
    if os.path.exists(local_path):
        _FEAT_MEM[name] = torch.load(local_path, weights_only=False)
        return _FEAT_MEM[name]
    print(f"extracting GQLA features for '{split}' ...")
    ds = ChunkedDocs(enc_docs, None, cfg, split=split, pack=False)   # extraction reads .items only
    feats = extract_chunk_features(GQLA.to(device), ds, use_layer=cfg.use_layer)
    torch.save(feats, local_path)
    shutil.copy(local_path, drive_path)                                        # persist to Drive once
    _FEAT_MEM[name] = feats
    return feats


In [15]:
import glob
# ============================================================
# 4. LOSSES, TRAINING, EVALUATION   (as before; adds PEFT optimiser groups, knowledge distillation, fp16 autocast and cost tracking)
# ============================================================
class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021): down-weights the many easy negatives of a multi-label problem."""

    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        xs_neg = (1 - p + self.clip).clamp(max=1) if self.clip > 0 else 1 - p
        los = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        pt = p * y + xs_neg * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(los * (1 - pt) ** gamma).sum() / logits.size(0)


def build_loss(cfg, y_tr):
    if cfg.loss == "bce":
        return nn.BCEWithLogitsLoss()
    if cfg.loss == "bce_pw":      # rare statutes get a larger positive weight: ((N-n_c)/n_c)^power, clipped
        pos = y_tr.sum(0)
        pw = np.clip(((len(y_tr) - pos) / np.maximum(pos, 1)) ** cfg.pw_power, 1.0, cfg.pw_max)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32))
    if cfg.loss == "asl":
        return AsymmetricLoss()
    raise ValueError(cfg.loss)


def forward_batch(model, batch, mode):
    cm = batch["chunk_mask"].to(device)
    if mode == "token":
        return model(batch["ids"].to(device), batch["tok_mask"].to(device), cm)
    return model(batch["feats"].to(device), cm)


def resolve_amp(cfg, mode):
    """cfg.amp: None -> on for 'token' mode (the GQLA forward runs every step), off for 'cached' (tiny head);
    True/False forces it. Always off when the self-test disabled half precision or there is no GPU."""
    want = (mode == "token") if getattr(cfg, "amp", None) is None else bool(cfg.amp)
    return want and AMP_DTYPE is not None and device.type == "cuda"


def amp_context(enabled):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=enabled)


@torch.no_grad()
def predict_logits(model, loader, mode, amp=False):
    model.eval()
    out = []
    for b in loader:
        with amp_context(amp):
            logits = forward_batch(model, b, mode)
        out.append(logits.float().cpu().numpy())
    return np.concatenate(out)


def predict_probs(model, loader, mode, amp=False):
    return 1.0 / (1.0 + np.exp(-np.clip(predict_logits(model, loader, mode, amp), -30, 30)))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_model(model, mode, train_loader, dev_loader, y_dev, cfg, y_tr, tag):
    loss_fn = build_loss(cfg, y_tr).to(device)
    peft = getattr(model, "peft", None)
    peft_ids = peft.param_ids() if peft is not None else set()
    new_p = [p for n, p in model.named_parameters() if p.requires_grad and id(p) in peft_ids]       # new PEFT params
    bb = [p for n, p in model.named_parameters()
          if p.requires_grad and n.startswith("backbone.") and id(p) not in peft_ids]               # pre-trained weights being tuned
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]  # BiRNN heads
    groups = ([dict(params=rest, lr=cfg.lr)] + ([dict(params=new_p, lr=cfg.lr_peft)] if new_p else [])
              + ([dict(params=bb, lr=cfg.lr_backbone)] if bb else []))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    total = cfg.epochs * math.ceil(len(train_loader) / cfg.accum)
    if peft is not None:
        peft.setup(total)
        print(f"[{tag}] {peft.describe()}")
    kd = getattr(cfg, "kd", None)                       # knowledge distillation: dict(teachers=[names], T=2.0, lam=0.5)
    if kd:
        t_logits = torch.from_numpy(cfg.kd_teacher_logits)          # [N_train, L], same order as the training docs
        print(f"[{tag}] distilling from {kd['teachers']} | T={kd['T']} lambda={kd['lam']}")
    gstep = 0
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))

    use_amp = resolve_amp(cfg, mode)
    # fp16 needs loss scaling; bf16 (Ampere+) does not. Disabled scaler = plain backward/step.
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and AMP_DTYPE == torch.float16)
    print(f"[{tag}] autocast: {AMP_DTYPE if use_amp else 'off'} | grad scaler: {scaler.is_enabled()}")

    best, best_state, bad = -1.0, None, 0
    for ep in range(cfg.epochs):
        model.train()
        t0, run, step = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            with amp_context(use_amp):
                logits = forward_batch(model, b, mode)
            logits, y = logits.float(), b["y"].to(device)               # losses always in fp32
            loss = loss_fn(logits, y)
            if kd:                                                      # (1-lam) * hard + lam * T^2 * soft (sigmoid targets)
                T = kd["T"]
                tl = t_logits[b["idx"]].to(device)
                soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(tl / T)) * (T * T)
                loss = (1 - kd["lam"]) * loss + kd["lam"] * soft
            if peft is not None:
                loss = loss + peft.extra_loss()                         # AdaLoRA orthogonality regulariser
            scaler.scale(loss / cfg.accum).backward()
            run += loss.item()
            if (i + 1) % cfg.accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(opt)
                if peft is not None:
                    peft.after_backward(gstep)                          # AdaLoRA: update importance scores
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                gstep += 1
                if peft is not None:
                    peft.after_step(gstep)                              # AdaLoRA: prune ranks to the budget schedule
            if ep == 0 and i + 1 == 20:                                 # cost estimate after 20 mini-batches
                sp = (time.time() - t0) / 20
                print(f"[{tag}] {sp:.2f} s/batch -> ~{sp*len(train_loader)/60:.1f} min per epoch")
        probs = predict_probs(model, dev_loader, mode, use_amp)
        thr, macro = tune_global_threshold(probs, y_dev)
        m = compute_metrics(y_dev, predict_labels(probs, thr))
        print(f"[{tag}] ep {ep+1:02d}  loss {run/len(train_loader):.4f}  dev macro-F1 {macro:.4f}  "
              f"micro-F1 {m['micro_f1']:.4f}  subset {m['subset_acc']:.4f}  thr {thr:.3f}  ({time.time()-t0:.0f}s)")
        if macro > best:
            best, bad = macro, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    model.load_state_dict(best_state, strict=False)           # frozen GQLA weights are untouched
    return best_state


BASE = dict(
    # long-document representation
    chunk_len=512, chunk_stride=512, max_chunks=16, chunk_strategy="head_tail",
    # GQLA features: use_layer=-1 -> all 10 blocks; k>=0 -> truncated GQLA of k+1 blocks (distillation student)
    use_layer=-1, unfreeze_top_k=0, gqla_chunk_bs=8,
    # bidirectional heads
    tok_rnn="gru", tok_hid=128, chunk_rnn="gru", chunk_hid=256, chunk_layers=1, pool="attn", dropout=0.3,
    # loss
    loss="bce_pw", pw_power=0.5, pw_max=30.0,
    # optimisation
    lr=1e-3, lr_backbone=2e-5, lr_peft=5e-4, wd=0.01, epochs=20, patience=5, batch_size=32, accum=1, warmup=0.06, clip=1.0,
    seed=42, min_support=3,
    # mixed precision: None = auto (on for mode='token', off for 'cached'); True/False forces it
    amp=None,
    # adaptation method for mode='token' (see 'PEFT for GQLA'): None | dict(method=..., **overrides);
    # distillation: dict(teachers=[...], T=2.0, lam=0.5)
    peft=None, kd=None,
)

RESULTS = {}
for _f in sorted(glob.glob(os.path.join(p5_dir, "result_*.pkl"))):          # survive a Colab disconnect / re-run
    RESULTS[os.path.basename(_f)[7:-4]] = pickle.load(open(_f, "rb"))
print("results already on Drive:", list(RESULTS))

_NO_EFFECT_KEYS = {"gqla_chunk_bs"}          # only changes micro-batching of the GQLA forward, not the result


def _same_cfg(a, b):
    return {k: v for k, v in a.items() if k not in _NO_EFFECT_KEYS} == {k: v for k, v in b.items() if k not in _NO_EFFECT_KEYS}


def run_experiment(name, mode="cached", keep_train_logits=False, force=False, **over):
    """mode='cached' : GQLA run once -> train BiRNN over chunk features (fast; no GQLA adaptation).
       mode='token'  : GQLA states -> token-level BiRNN -> chunk-level BiRNN; cfg.peft adapts the GQLA itself.
       keep_train_logits=True stores the model's logits on the TRAIN documents (needed if it will be a KD teacher)."""
    cfg = SimpleNamespace(**{**BASE, **over})
    old = RESULTS.get(name)
    if (not force and old is not None and old.get("mode") == mode and _same_cfg(old["cfg"], vars(cfg))
            and (not keep_train_logits or "train_logits" in old)):
        print(f"[{name}] already done with the same settings -> reusing saved result (force=True to retrain)")
        return old
    set_seed(cfg.seed)
    if cfg.peft and mode != "token":
        raise ValueError("peft adapts the GQLA itself -> use mode='token'")
    if mode == "cached":
        ftr, fdv, fte = (get_features(s, e, cfg) for s, e in
                         (("train", enc_train), ("dev", enc_dev), ("test", enc_test)))
        d_tr, d_dv, d_te = FeatDocs(ftr, y_train), FeatDocs(fdv, y_dev), FeatDocs(fte, None)
        coll, model = collate_feats, CachedFeatureClassifier(ftr[0].size(1), L, cfg)
        g = None
    else:
        d_tr, d_dv, d_te = (ChunkedDocs(enc_train, y_train, cfg, "train"), ChunkedDocs(enc_dev, y_dev, cfg, "dev"),
                            ChunkedDocs(enc_test, None, cfg, "test"))
        g = copy.deepcopy(GQLA) if (cfg.peft or cfg.unfreeze_top_k > 0) else GQLA   # never modify the shared Phase-1 model
        coll, model = collate_tokens, GQLABiRNNClassifier(g, L, cfg)
    if cfg.kd:                                                       # teacher logits on the training docs (averaged)
        miss = [t for t in cfg.kd["teachers"] if "train_logits" not in RESULTS.get(t, {})]
        assert not miss, f"teacher(s) {miss}: run them first with keep_train_logits=True"
        cfg.kd_teacher_logits = np.mean([RESULTS[t]["train_logits"] for t in cfg.kd["teachers"]], axis=0).astype(np.float32)
    model.to(device)
    mk = lambda d, sh: DataLoader(d, batch_size=cfg.batch_size if sh else max(cfg.batch_size, 8), shuffle=sh,
                                  collate_fn=coll, num_workers=0, drop_last=False)
    tr, dv, te, tr_eval = mk(d_tr, True), mk(d_dv, False), mk(d_te, False), mk(d_tr, False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_used = model.backbone.n_used if mode == "token" else (len(GQLA.layers) if cfg.use_layer == -1 else cfg.use_layer + 1)
    params_M, size_MB = model_footprint(model, mode, n_used)
    print(f"\n=== {name} | mode={mode} | trainable params {n_train/1e6:.3f}M | deployed pipeline {params_M:.1f}M params "
          f"/ {size_MB:.0f} MB ({n_used} GQLA blocks) ===")

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_model(model, mode, tr, dv, y_dev, cfg, y_train, name)
    train_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0

    amp_on = resolve_amp(cfg, mode)
    t0 = time.time()
    lg_dev = predict_logits(model, dv, mode, amp_on)
    infer_s = time.time() - t0                                        # dev-set inference wall-clock (incl. GQLA forward in token mode)
    p_dev = 1.0 / (1.0 + np.exp(-np.clip(lg_dev, -30, 30)))
    p_test = predict_probs(model, te, mode, amp_on)
    g_thr, _ = tune_global_threshold(p_dev, y_dev)
    pc_thr = tune_per_class_threshold(p_dev, y_dev, g_thr, min_support=cfg.min_support)
    res = dict(cfg={k: v for k, v in vars(cfg).items() if k != "kd_teacher_logits"}, mode=mode, p_dev=p_dev, p_test=p_test,
               g_thr=g_thr, pc_thr=pc_thr, trainable_params=n_train, deployed_params_M=params_M, deployed_size_MB=size_MB,
               train_seconds=train_s, dev_infer_seconds=infer_s, peak_gpu_MB=peak_mb,
               m_fixed=compute_metrics(y_dev, predict_labels(p_dev, 0.5)),
               m_global=compute_metrics(y_dev, predict_labels(p_dev, g_thr)),
               m_perclass=compute_metrics(y_dev, predict_labels(p_dev, pc_thr)),
               honest=honest_threshold_eval(p_dev, y_dev))
    if getattr(model, "peft", None) is not None and model.peft.allocator is not None:
        res["adalora_ranks"] = model.peft.allocator.effective_ranks()
    if keep_train_logits:
        res["train_logits"] = predict_logits(model, tr_eval, mode, amp_on)
    RESULTS[name] = res
    pickle.dump({k: v for k, v in res.items()}, open(os.path.join(p5_dir, f"result_{name}.pkl"), "wb"))
    print(f"[{name}] dev macro-F1  thr=0.5: {res['m_fixed']['macro_f1']:.4f} | global thr {g_thr:.3f}: "
          f"{res['m_global']['macro_f1']:.4f} | per-class: {res['m_perclass']['macro_f1']:.4f} (optimistic)")
    print(f"[{name}] honest split-half macro-F1 (tune on half, score on other half): {res['honest']}")
    print(f"[{name}] cost: train {train_s/60:.1f} min | dev inference {infer_s:.1f}s | peak GPU {peak_mb:.0f} MB")
    del model, g
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res


def write_submission(name, path, per_class=False):
    r = RESULTS[name]
    thr = r["pc_thr"] if per_class else r["g_thr"]
    pred = predict_labels(r["p_test"], thr)
    out = [dict(id=rec["id"], pred=sorted(int(idx2lab[j]) for j in np.flatnonzero(p)))
           for rec, p in zip(test_recs, pred)]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one prediction per test id"
    assert all(len(o["pred"]) >= 1 and len(set(o["pred"])) == len(o["pred"]) for o in out)
    json.dump(out, open(path, "w"), ensure_ascii=False)
    print(f"wrote {len(out)} predictions -> {path}  (avg {np.mean([len(o['pred']) for o in out]):.2f} statutes/doc)")

results already on Drive: ['E0_cached_baseline', 'E2_partial_finetune', 'cached_bigru', 'e1_lorma_plus', 'e2_adalora', 'e4_adapter', 'e5_partial_ft']


In [16]:
# ============================================================
# 5. RUN  -  the six Phase-5 experiments for the GQLA architecture
#
#   baseline  frozen GQLA + BiGRU on cached features                (zero adaptation of the GQLA; NOT one of the six)
#   E1  LoRMA+            multiplicative low-rank adaptation of W_UQ / W_UV (query / value up-projections)  [compulsory technique]
#   E2  AdaLoRA           SVD-style low-rank update with importance-based rank pruning
#   E3  QLoRA             frozen 4-bit NF4 GQLA + LoRA
#   E4  Adapters          bottleneck adapters after attention and after the FFN (Houlsby)
#   E5  Partial FT        fine-tune the top-3 GQLA blocks (layer-wise fine-tuning)
#   E6  Knowledge dist.   teacher = ensemble of E1-E5 (10-block GQLA + adaptation), student = 5-block GQLA + small BiGRU
#
# That is 5 different fine-tuning / PEFT techniques + 1 distillation experiment (brief, sec. 11.2).
# Optional extras below: LoRA, LoRMA-pi, prompt tuning, full fine-tuning, student without distillation.
#
# !! Every number below (epochs, ranks, learning rates, chunks, temperature, ...) is only a STARTING POINT so that the
# !! notebook runs end-to-end. Tuning them and interpreting the results is your job (and the course rules say so).
# !! Run ONE experiment per cell first and read the "s/batch -> ~N min per epoch" line before queueing the rest.
# ============================================================
import sys, gc

TOK = dict(mode="token", batch_size=4, accum=4, max_chunks=8, epochs=4, patience=2,
           lr=1e-3, lr_peft=5e-4, lr_backbone=2e-5, gqla_chunk_bs=32)

_GOOD_CBS = {}       # method -> largest GQLA chunk micro-batch that fitted, so later runs skip the OOM retries


def run_safe(name, chunk_bs_ladder=(32, 16, 8, 4, 2, 1), **kw):
    """run_experiment, retried with a smaller GQLA micro-batch of chunks on CUDA out-of-memory (the numbers of the
       experiment do not change, only the micro-batching of the forward pass). A batch of 4 docs x 8 chunks = 32 chunks,
       so gqla_chunk_bs=32 runs the GQLA once per step instead of 4 times when it fits in memory."""
    kw.pop("gqla_chunk_bs", None)
    key = (kw.get("peft") or {}).get("method")
    ladder = [c for c in chunk_bs_ladder if c <= _GOOD_CBS.get(key, chunk_bs_ladder[0])]
    for cbs in ladder:
        gc.collect(); torch.cuda.empty_cache()
        oom = False
        try:
            r = run_experiment(name, gqla_chunk_bs=cbs, **kw)
            _GOOD_CBS[key] = cbs
            return r
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:                                            # leave the except block BEFORE cleaning so the traceback frees its tensors
            sys.last_traceback = sys.last_value = sys.last_type = None
            gc.collect(); torch.cuda.empty_cache()
            print(f"[{name}] OOM with gqla_chunk_bs={cbs}; retrying smaller")
    raise RuntimeError(f"{name}: still out of memory at gqla_chunk_bs={ladder[-1]}; reduce max_chunks or batch_size "
                       "(note: that changes the experiment)")


In [17]:
# ---- baseline: frozen GQLA, cached features (fast; shows what the PEFT methods have to beat) -----------------
r_base = run_experiment("cached_bigru", mode="cached")


[cached_bigru] already done with the same settings -> reusing saved result (force=True to retrain)


In [18]:
# ---- E1  LoRMA+  (compulsory) -------------------------------------------------------------------------------------
r1 = run_safe("e1_lorma_plus", keep_train_logits=True,
                    peft=dict(method="lorma_plus", r=8, alpha=16, top_blocks=4, targets=("W_UQ", "W_UV")), **TOK)


[e1_lorma_plus] already done with the same settings -> reusing saved result (force=True to retrain)


In [19]:
# ---- E2  AdaLoRA --------------------------------------------------------------------------------------------------
r2 = run_safe("e2_adalora", keep_train_logits=True,
                    peft=dict(method="adalora", init_r=12, target_r=6, top_blocks=4), **TOK)


[e2_adalora] already done with the same settings -> reusing saved result (force=True to retrain)


In [26]:
# ---- E3  QLoRA  (4-bit NF4 base for all 10 blocks, LoRA in the top 4 blocks) ----------------------------------------
r3 = run_safe("e3_qlora", keep_train_logits=True,
                    peft=dict(method="qlora", r=16, alpha=32, top_blocks=4), **TOK)


[e3_qlora] already done with the same settings -> reusing saved result (force=True to retrain)


In [21]:
# ---- E4  Bottleneck adapters ----------------------------------------------------------------------------------------
r4 = run_safe("e4_adapter", keep_train_logits=True,
                    peft=dict(method="adapter", bottleneck=64, top_blocks=4), **TOK)


[e4_adapter] already done with the same settings -> reusing saved result (force=True to retrain)


In [22]:
# ---- E5  Partial / layer-wise fine-tuning (pre-trained weights of the top 3 blocks + final norm are updated) -------------------------
r5 = run_safe("e5_partial_ft", keep_train_logits=True,
                    peft=dict(method="partial", k=3), **TOK)


[e5_partial_ft] already done with the same settings -> reusing saved result (force=True to retrain)


In [23]:
# ---- E6  Knowledge distillation -------------------------------------------------------------------------------------
# Teacher : ensemble (mean logit) of E1-E5 = 10 GQLA blocks + adaptation + hierarchical BiRNN head.
# Student : the FIRST 5 GQLA blocks (use_layer=4, parameter-free LayerNorm on top) frozen + a smaller BiGRU head (chunk_hid=128);
#           features cached. -> 5/10 of the GQLA compute and size at inference.
#           Loss = (1-lam)*weighted-BCE + lam*T^2*BCE(student/T, sigmoid(teacher/T)).
r6 = run_experiment("e6_kd_student", mode="cached", use_layer=4, chunk_hid=128,
                    kd=dict(teachers=["e1_lorma_plus", "e2_adalora", "e3_qlora", "e4_adapter", "e5_partial_ft"], T=2.0, lam=0.5))



=== e6_kd_student | mode=cached | trainable params 0.752M | deployed pipeline 52.7M params / 211 MB (5 GQLA blocks) ===
[e6_kd_student] distilling from ['e1_lorma_plus', 'e2_adalora', 'e3_qlora', 'e4_adapter', 'e5_partial_ft'] | T=2.0 lambda=0.5
[e6_kd_student] autocast: off | grad scaler: False
[e6_kd_student] 0.02 s/batch -> ~0.5 min per epoch
[e6_kd_student] ep 01  loss 1.1950  dev macro-F1 0.1045  micro-F1 0.1931  subset 0.0000  thr 0.125  (17s)
[e6_kd_student] ep 02  loss 1.1080  dev macro-F1 0.1456  micro-F1 0.2119  subset 0.0000  thr 0.150  (16s)
[e6_kd_student] ep 03  loss 1.0821  dev macro-F1 0.1845  micro-F1 0.2959  subset 0.0002  thr 0.225  (16s)
[e6_kd_student] ep 04  loss 1.0640  dev macro-F1 0.2087  micro-F1 0.3066  subset 0.0032  thr 0.250  (16s)
[e6_kd_student] ep 05  loss 1.0501  dev macro-F1 0.2291  micro-F1 0.3396  subset 0.0057  thr 0.300  (17s)
[e6_kd_student] ep 06  loss 1.0392  dev macro-F1 0.2438  micro-F1 0.3510  subset 0.0047  thr 0.300  (16s)
[e6_kd_student]

In [24]:
# ---- optional extras --------------------------------------------------------------------------------------------------
# r6b = run_experiment("e6b_student_no_kd", mode="cached", use_layer=4, chunk_hid=128)        # same student, no teacher (isolates the KD effect)
# r7  = run_safe("x_lorma_pi", keep_train_logits=True, peft=dict(method="lorma_pi", r=8, top_blocks=4), **TOK)
# r7b = run_safe("x_lora", keep_train_logits=True, peft=dict(method="lora", r=8, alpha=16, top_blocks=4), **TOK)      # additive LoRA (the baseline LoRMA is compared with)
# r8  = run_safe("x_prompt", peft=dict(method="prompt", n_tokens=20), **{**TOK, "lr_peft": 5e-2})     # backward through ALL 10 blocks: slow
# r9  = run_safe("x_full_ft", peft=dict(method="full"), **{**TOK, "batch_size": 2, "accum": 8})        # all blocks + embeddings: slow, needs memory
# Other levers from the brief on the same code: loss=("bce"|"bce_pw"|"asl"), chunk_strategy, max_chunks, pool="label_attn",
# chunk_rnn=("gru"|"lstm"|"transformer"), use_layer, and the thresholds (global / per class) in the summary below.


In [27]:
# ---- summary table (accuracy + cost) ------------------------------------------------------------------------------------------
print(f"\n{'experiment':20s} {'macro-F1':>8s} {'micro-F1':>8s} {'subset':>7s} {'honest(g)':>9s} {'train M':>8s} "
      f"{'deploy M':>8s} {'MB':>6s} {'train min':>9s} {'dev inf s':>9s} {'peak MB':>8s}")
for n, r in RESULTS.items():
    m = r["m_global"]
    # Use .get() with default fallbacks to handle older runs that lack deployment size metrics
    train_m = r.get("trainable_params", 0) / 1e6
    deploy_m = r.get("deployed_params_M", 0.0)
    size_mb = r.get("deployed_size_MB", 0.0)
    train_min = r.get("train_seconds", 0.0) / 60
    dev_inf = r.get("dev_infer_seconds", 0.0)
    peak_gpu = r.get("peak_gpu_MB", 0.0)
    honest_g = r.get("honest", {}).get("global", 0.0)

    print(f"{n:20s} {m['macro_f1']:8.4f} {m['micro_f1']:8.4f} {m['subset_acc']:7.4f} {honest_g:9.4f} "
          f"{train_m:8.3f} {deploy_m:8.1f} {size_mb:6.0f} "
          f"{train_min:9.1f} {dev_inf:9.1f} {peak_gpu:8.0f}")
if "e2_adalora" in RESULTS:
    print("AdaLoRA final ranks per adapted matrix:", RESULTS["e2_adalora"].get("adalora_ranks", "N/A"))

# ---- rare-statute analysis for one run ----------------------------------------------------------------------------------
best_name = max(RESULTS, key=lambda n: RESULTS[n].get("honest", {}).get("global", -1.0))
r = RESULTS[best_name]
print(f"\nF1 by number of TRAIN docs per statute ({best_name}):")
for bucket, n_cls, f1 in rare_report(y_dev, predict_labels(r["p_dev"], r["g_thr"]), y_train):
    print(f"  train-freq {bucket:>8s}: {n_cls:4d} statutes  mean F1 {f1:.3f}")


experiment           macro-F1 micro-F1  subset honest(g)  train M deploy M     MB train min dev inf s  peak MB
E0_cached_baseline     0.3269   0.4318  0.0341    0.3255    2.156      0.0      0       0.0       0.0        0
E2_partial_finetune    0.1658   0.3198  0.0003    0.1649   16.191      0.0      0       0.0       0.0        0
cached_bigru           0.3269   0.4318  0.0341    0.3255    2.156     89.3    357       5.7       1.5      480
e1_lorma_plus          0.3295   0.4457  0.0323    0.3283    2.517     89.6    359     303.1     470.3     1607
e2_adalora             0.3397   0.4544  0.0404    0.3388    3.148     90.3    361     362.8     501.1     1598
e4_adapter             0.3364   0.4475  0.0288    0.3350    2.652     89.8    359     266.1     437.1     1559
e5_partial_ft          0.3411   0.4578  0.0380    0.3389   23.224     89.2    357     259.0     431.8     1789
e3_qlora               0.3409   0.4564  0.0345    0.3391    3.489     90.6    119     329.2     486.0     2678


In [28]:
# ---- Codabench packages: prediction.json at the ROOT of submission_<name>.zip -------------------------------------------------
import zipfile


def make_submission_zip(name, per_class=None):
    """Writes prediction.json for experiment `name` and zips it (flat, as Codabench expects).
       per_class=None -> use per-class thresholds only if their honest split-half score beats the global threshold."""
    r = RESULTS[name]
    use_pc = (r["honest"]["per_class"] > r["honest"]["global"]) if per_class is None else per_class
    d = os.path.join(p5_dir, f"sub_{name}"); os.makedirs(d, exist_ok=True)
    write_submission(name, os.path.join(d, "prediction.json"), per_class=use_pc)
    zpath = os.path.join(p5_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), arcname="prediction.json")
    print(f"  -> {zpath}")
    return zpath


# One zip per experiment (the brief needs >= 6 distinct, non-identical submissions):
for name in RESULTS:
    make_submission_zip(name)


wrote 13019 predictions -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/sub_E0_cached_baseline/prediction.json  (avg 4.91 statutes/doc)
  -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/submission_E0_cached_baseline.zip
wrote 13019 predictions -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/sub_E2_partial_finetune/prediction.json  (avg 8.76 statutes/doc)
  -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/submission_E2_partial_finetune.zip
wrote 13019 predictions -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/sub_cached_bigru/prediction.json  (avg 4.91 statutes/doc)
  -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/submission_cached_bigru.zip
wrote 13019 predictions -> /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/phase5/sub_e1_lorma_plus/prediction.json  (avg 4.96 statutes/doc)
  -> /content/drive/MyD